In [5]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))


TensorFlow version: 2.20.0
Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import files

uploaded = files.upload()

Saving repeated_stratified_5fold_assignments.csv to repeated_stratified_5fold_assignments.csv


In [3]:
import shutil
import tarfile
from pathlib import Path


# ============================================================
# RESTORE THE PROJECT DATASET TO THE COLAB RUNTIME
# ============================================================

# Create the path to the project archive stored in Google Drive
DRIVE_ARCHIVE = Path(
    "/content/drive/MyDrive/"
    "brain_tumour_colab/"
    "brain_tumour_colab_bundle.tar"
)


# Create the local path where the archive will temporarily be copied
LOCAL_ARCHIVE = Path("/content/brain_tumour_colab_bundle.tar")


# Create the path where the project will be restored
PROJECT_ROOT = Path("/content/brain-tumour-mri-classification")


print("=" * 70)
print("RESTORE PROJECT DATA")
print("=" * 70)
print("Drive archive:", DRIVE_ARCHIVE)
print("Drive archive exists:", DRIVE_ARCHIVE.exists())
print("Project root:", PROJECT_ROOT)


# Stop if the project archive cannot be found in Google Drive
if not DRIVE_ARCHIVE.exists():
    raise FileNotFoundError(f"Archive not found: {DRIVE_ARCHIVE}")


# Restore the project only if it is not already present in the runtime
if not PROJECT_ROOT.exists():

    print("\nCopying archive to Colab runtime...")
    shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)


    # Create the project directory
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


    print("Extracting archive...")

    with tarfile.open(LOCAL_ARCHIVE, "r") as archive:
        archive.extractall(PROJECT_ROOT, filter="data")

else:
    print("\nProject directory already exists. Skipping archive extraction.")


# ============================================================
# VERIFY THE RESTORED PROJECT
# ============================================================

# Create the path to the fixed five-fold cross-validation file
FOLDS_FILE = PROJECT_ROOT / "splits" / "five_fold_cross_validation.csv"


# Create the path to the cropped Training and Testing dataset
DATA_DIR = PROJECT_ROOT / "processed_data_cropped"


# Count every PNG image in the cropped dataset
png_count = len(list(DATA_DIR.rglob("*.png")))

print("\n--- Verification ---")
print("Project root exists:", PROJECT_ROOT.exists())
print("Fold file exists:", FOLDS_FILE.exists())
print("Dataset folder exists:", DATA_DIR.exists())
print("PNG images found:", png_count)


# Stop if the project was not restored correctly
if not FOLDS_FILE.exists():
    raise FileNotFoundError(f"Five-fold cross-validation file not found: {FOLDS_FILE}")


if not DATA_DIR.exists():
    raise FileNotFoundError(f"Cropped dataset folder not found: {DATA_DIR}")


if png_count != 7198:
    raise RuntimeError("Expected 7,198 PNG images, "f"but found {png_count}.")

print("\nProject restored successfully.")

RESTORE PROJECT DATA
Drive archive: /content/drive/MyDrive/brain_tumour_colab/brain_tumour_colab_bundle.tar
Drive archive exists: True
Project root: /content/brain-tumour-mri-classification

Copying archive to Colab runtime...
Extracting archive...

--- Verification ---
Project root exists: True
Fold file exists: True
Dataset folder exists: True
PNG images found: 7198

Project restored successfully.


In [4]:
# ============================================================
# VALIDATE THE REPEATED 10 x 5 CROSS-VALIDATION ASSIGNMENTS
# ============================================================

from pathlib import Path
import shutil
import pandas as pd


UPLOADED_SPLIT_FILE = Path(
    "/content/repeated_stratified_5fold_assignments.csv"
)

REPEATED_SPLITS_DIR = (
    PROJECT_ROOT
    / "repeated_cv"
    / "splits"
)

REPEATED_SPLITS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPEATED_FOLDS_FILE = (
    REPEATED_SPLITS_DIR
    / "repeated_stratified_5fold_assignments.csv"
)


# Copy the uploaded file into the restored project structure
shutil.copy2(
    UPLOADED_SPLIT_FILE,
    REPEATED_FOLDS_FILE,
)


# Load the assignments
repeated_folds_df = pd.read_csv(
    REPEATED_FOLDS_FILE
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(repeated_folds_df) == 56000

assert repeated_folds_df["repeat"].nunique() == 10

assert set(
    repeated_folds_df["repeat"]
) == set(range(1, 11))

assert set(
    repeated_folds_df["fold"]
) == {1, 2, 3, 4, 5}

assert (
    repeated_folds_df
    .groupby(
        ["repeat", "fold"]
    )
    .size()
    .eq(1120)
    .all()
)

assert (
    repeated_folds_df
    .groupby(
        ["repeat", "fold", "class"]
    )
    .size()
    .eq(280)
    .all()
)

assert (
    repeated_folds_df[
        "relative_path"
    ]
    .str.startswith("Training/")
    .all()
)


print(
    "Repeated split file:",
    REPEATED_FOLDS_FILE,
)

print(
    "Rows:",
    len(repeated_folds_df),
)

print(
    "Repeats:",
    repeated_folds_df[
        "repeat"
    ].nunique(),
)

print(
    "Folds per repeat:",
    repeated_folds_df[
        "fold"
    ].nunique(),
)

print(
    "\nSplit seeds:"
)

print(
    repeated_folds_df[
        ["repeat", "split_seed"]
    ]
    .drop_duplicates()
    .sort_values("repeat")
    .to_string(index=False)
)

print(
    "\nRepeated 10 x 5 split validation PASSED."
)

Repeated split file: /content/brain-tumour-mri-classification/repeated_cv/splits/repeated_stratified_5fold_assignments.csv
Rows: 56000
Repeats: 10
Folds per repeat: 5

Split seeds:
 repeat  split_seed
      1      202601
      2      202602
      3      202603
      4      202604
      5      202605
      6      202606
      7      202607
      8      202608
      9      202609
     10      202610

Repeated 10 x 5 split validation PASSED.


In [5]:
# ============================================================
# Step 5. REPEATED NESTED-CV EFFICIENTNETB0 EXPERIMENT SETUP
# ============================================================

import gc
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATA_DIR = (
    PROJECT_ROOT
    / "processed_data_cropped"
)

assert DATA_DIR.exists(), (
    "Processed dataset directory was not found:\n"
    f"{DATA_DIR}"
)


# ------------------------------------------------------------
# Persistent Google Drive results
# ------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/brain_tumour_colab"
)

REPEATED_EFFICIENTNETB0_DIR = (
    DRIVE_ROOT
    / "results"
    / "repeated_nested_cv"
    / "efficientnetb0"
)

REPEATED_EFFICIENTNETB0_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Image / class configuration
# ------------------------------------------------------------

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary",
]

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name
    in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for class_name, index
    in CLASS_TO_INDEX.items()
}


# ------------------------------------------------------------
# Repeated nested-CV configuration
# ------------------------------------------------------------

NUMBER_OF_REPEATS = 10
NUMBER_OF_OUTER_FOLDS = 5

INNER_VALIDATION_FRACTION = 0.10


# ------------------------------------------------------------
# Deterministic seed schedules
#
# The internal split schedule is intentionally the SAME as
# ResNet50 so that both CNN architectures use the same 90/10
# internal partition within each corresponding outer fold.
#
# EfficientNetB0 receives its own model/training seed schedule.
# ------------------------------------------------------------

BASE_MODEL_SEED = 707000
BASE_INNER_SPLIT_SEED = 606000


def get_efficientnet_model_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_MODEL_SEED
        + repeat_number * 100
        + fold_number
    )


def get_efficientnet_inner_split_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_INNER_SPLIT_SEED
        + repeat_number * 100
        + fold_number
    )


# ------------------------------------------------------------
# TensorFlow reproducibility
# ------------------------------------------------------------

try:
    tf.config.experimental.enable_op_determinism()
    determinism_status = "enabled"

except Exception as error:
    determinism_status = (
        f"requested but unavailable: {error}"
    )


# Keep EfficientNetB0 training in float32, matching the
# original EfficientNetB0 experiment.
tf.keras.backend.set_floatx(
    "float32"
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

gpu_devices = (
    tf.config.list_physical_devices(
        "GPU"
    )
)

assert gpu_devices, (
    "No GPU detected."
)

assert len(
    repeated_folds_df
) == 56000, (
    "Expected 56,000 repeated-fold assignment rows."
)

assert NUMBER_OF_REPEATS == 10

assert NUMBER_OF_OUTER_FOLDS == 5

assert INNER_VALIDATION_FRACTION == 0.10


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("=" * 70)
print("REPEATED NESTED-CV EFFICIENTNETB0 SETUP")
print("=" * 70)

print(
    "GPU:",
    gpu_devices,
)

print(
    "Dataset:",
    DATA_DIR,
)

print(
    "Results directory:",
    REPEATED_EFFICIENTNETB0_DIR,
)

print(
    "\nRepeats:",
    NUMBER_OF_REPEATS,
)

print(
    "Outer folds per repeat:",
    NUMBER_OF_OUTER_FOLDS,
)

print(
    "Total outer evaluations:",
    NUMBER_OF_REPEATS
    * NUMBER_OF_OUTER_FOLDS,
)

print(
    "Internal validation fraction:",
    INNER_VALIDATION_FRACTION,
)

print(
    "\nRepeat 1 / Fold 1 model seed:",
    get_efficientnet_model_seed(
        1,
        1,
    ),
)

print(
    "Repeat 1 / Fold 1 inner-split seed:",
    get_efficientnet_inner_split_seed(
        1,
        1,
    ),
)

print(
    "\nTensorFlow deterministic operations:",
    determinism_status,
)

print(
    "\nEfficientNetB0 repeated-CV setup PASSED."
)

REPEATED NESTED-CV EFFICIENTNETB0 SETUP
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Dataset: /content/brain-tumour-mri-classification/processed_data_cropped
Results directory: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0

Repeats: 10
Outer folds per repeat: 5
Total outer evaluations: 50
Internal validation fraction: 0.1

Repeat 1 / Fold 1 model seed: 707101
Repeat 1 / Fold 1 inner-split seed: 606101

TensorFlow deterministic operations: enabled

EfficientNetB0 repeated-CV setup PASSED.


In [6]:
# ============================================================
# Step 6. PREPARE REPEATED EFFICIENTNETB0 ASSIGNMENTS
# ============================================================

# Work from the already validated shared 10 x 5 split table
efficientnet_repeated_assignments = (
    repeated_folds_df
    .copy()
)


# Add the fixed numeric class index
efficientnet_repeated_assignments[
    "class_index"
] = (
    efficientnet_repeated_assignments[
        "class"
    ]
    .map(
        CLASS_TO_INDEX
    )
)


# Add the full image path
efficientnet_repeated_assignments[
    "image_path"
] = (
    efficientnet_repeated_assignments[
        "relative_path"
    ]
    .apply(
        lambda relative_path:
            DATA_DIR
            / Path(relative_path)
    )
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(
    efficientnet_repeated_assignments
) == 56000


assert (
    efficientnet_repeated_assignments[
        "class_index"
    ]
    .notna()
    .all()
)


assert (
    efficientnet_repeated_assignments[
        "relative_path"
    ]
    .str.startswith("Training/")
    .all()
)


assert all(
    image_path.exists()
    for image_path
    in efficientnet_repeated_assignments[
        "image_path"
    ]
)


# Verify that the numeric labels stored in the shared split
# file agree exactly with the fixed class mapping.
assert (
    efficientnet_repeated_assignments[
        "class_index"
    ]
    .astype(int)
    .to_numpy()
    ==
    efficientnet_repeated_assignments[
        "label"
    ]
    .astype(int)
    .to_numpy()
).all()


# Every repeat must contain the same 5,600 unique images
images_per_repeat = (
    efficientnet_repeated_assignments
    .groupby("repeat")[
        "relative_path"
    ]
    .nunique()
)


assert (
    images_per_repeat
    == 5600
).all()


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "Rows:",
    len(
        efficientnet_repeated_assignments
    ),
)

print(
    "Unique images per repeat:"
)

print(
    images_per_repeat.to_dict()
)

print(
    "\nClass mapping:",
    CLASS_TO_INDEX,
)

print(
    "\nShared labels match class mapping."
)

print(
    "\nMissing image files: 0"
)

print(
    "\nRepeated EfficientNetB0 assignment preparation PASSED."
)

Rows: 56000
Unique images per repeat:
{1: 5600, 2: 5600, 3: 5600, 4: 5600, 5: 5600, 6: 5600, 7: 5600, 8: 5600, 9: 5600, 10: 5600}

Class mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}

Shared labels match class mapping.

Missing image files: 0

Repeated EfficientNetB0 assignment preparation PASSED.


In [7]:
# ============================================================
# STEP 7: CORRECT EFFICIENTNETB0 PREPROCESSING AND DATASET FUNCTIONS
# ============================================================

def load_and_preprocess_efficientnetb0_image(
    image_path,
    class_index,
):
    """
    Load one stored 224x224 grayscale PNG and prepare it
    for ImageNet-pretrained EfficientNetB0.

    IMPORTANT:
    EfficientNetB0 in tf.keras includes its own input
    rescaling inside the model. Therefore the image must
    remain float32 in the original 0-255 pixel range.

    No external preprocess_input function is applied.
    """

    # Read image file
    image_bytes = tf.io.read_file(
        image_path
    )

    # Decode as one-channel grayscale
    grayscale_image = tf.io.decode_png(
        image_bytes,
        channels=1,
    )

    grayscale_image = tf.ensure_shape(
        grayscale_image,
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            1,
        ),
    )

    # Convert grayscale -> 3-channel pseudo-RGB
    pseudo_rgb_image = (
        tf.image.grayscale_to_rgb(
            grayscale_image
        )
    )

    pseudo_rgb_image = tf.ensure_shape(
        pseudo_rgb_image,
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS,
        ),
    )

    # Preserve the original EfficientNetB0 preprocessing:
    # convert to float32 but KEEP values in the 0-255 range.
    efficientnetb0_image = tf.cast(
        pseudo_rgb_image,
        tf.float32,
    )

    efficientnetb0_image = tf.ensure_shape(
        efficientnetb0_image,
        (
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS,
        ),
    )

    class_index = tf.cast(
        class_index,
        tf.int32,
    )

    return (
        efficientnetb0_image,
        class_index,
    )


def make_efficientnetb0_dataset(
    dataframe,
    training,
    seed,
    batch_size,
):
    """
    Build a deterministic TensorFlow dataset for
    EfficientNetB0.
    """

    image_paths = (
        dataframe[
            "image_path"
        ]
        .astype(str)
        .to_numpy()
    )

    class_indices = (
        dataframe[
            "class_index"
        ]
        .astype(np.int32)
        .to_numpy()
    )

    dataset = (
        tf.data.Dataset
        .from_tensor_slices(
            (
                image_paths,
                class_indices,
            )
        )
    )

    options = tf.data.Options()

    options.experimental_deterministic = True

    dataset = dataset.with_options(
        options
    )

    # Shuffle Training data only.
    if training:

        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=seed,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        load_and_preprocess_efficientnetb0_image,
        num_parallel_calls=tf.data.AUTOTUNE,
        deterministic=True,
    )

    dataset = dataset.batch(
        batch_size,
        drop_remainder=False,
    )

    dataset = dataset.prefetch(
        tf.data.AUTOTUNE
    )

    return dataset


# ------------------------------------------------------------
# Test preprocessing on one actual Training image
# ------------------------------------------------------------

test_row = (
    efficientnet_repeated_assignments
    .iloc[0]
)

test_image, test_label = (
    load_and_preprocess_efficientnetb0_image(
        str(
            test_row[
                "image_path"
            ]
        ),
        int(
            test_row[
                "class_index"
            ]
        ),
    )
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert test_image.shape == (
    224,
    224,
    3,
)

assert (
    test_image.dtype
    == tf.float32
)

assert (
    test_label.dtype
    == tf.int32
)

assert bool(
    tf.reduce_all(
        tf.math.is_finite(
            test_image
        )
    )
)

test_minimum = float(
    tf.reduce_min(
        test_image
    )
)

test_maximum = float(
    tf.reduce_max(
        test_image
    )
)

assert test_minimum >= 0.0

assert test_maximum <= 255.0

assert (
    INDEX_TO_CLASS[
        int(
            test_label.numpy()
        )
    ]
    ==
    test_row[
        "class"
    ]
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print(
    "Test image shape:",
    test_image.shape,
)

print(
    "Test image dtype:",
    test_image.dtype,
)

print(
    "Value range:",
    test_minimum,
    "to",
    test_maximum,
)

print(
    "Test label:",
    int(
        test_label.numpy()
    ),
)

print(
    "Decoded class:",
    INDEX_TO_CLASS[
        int(
            test_label.numpy()
        )
    ],
)

print(
    "Expected class:",
    test_row[
        "class"
    ],
)

print(
    "All values finite:",
    bool(
        tf.reduce_all(
            tf.math.is_finite(
                test_image
            )
        )
    ),
)

print(
    "\nNo external EfficientNetB0 preprocess_input applied."
)

print(
    "Input remains float32 in the 0-255 range."
)

print(
    "\nStep 7 preprocessing validation PASSED."
)

Test image shape: (224, 224, 3)
Test image dtype: <dtype: 'float32'>
Value range: 0.0 to 202.0
Test label: 0
Decoded class: glioma
Expected class: glioma
All values finite: True

No external EfficientNetB0 preprocess_input applied.
Input remains float32 in the 0-255 range.

Step 7 preprocessing validation PASSED.


In [8]:
# ============================================================
# STEP 8: CREATE THE 90/10 INTERNAL VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import StratifiedShuffleSplit


def make_repeated_efficientnet_partitions(
    repeat_number,
    fold_number,
):
    """
    For one repeated outer fold, create:

    - 4,032 model-training images
    -   448 internal-validation images
    - 1,120 untouched outer-validation images
    """

    # --------------------------------------------------------
    # Get the 5,600 Training images for this repeat
    # --------------------------------------------------------

    repeat_assignments = (
        efficientnet_repeated_assignments[
            efficientnet_repeated_assignments["repeat"]
            == repeat_number
        ]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Outer validation = the selected outer fold
    # --------------------------------------------------------

    outer_validation = (
        repeat_assignments[
            repeat_assignments["fold"]
            == fold_number
        ]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Outer training = the other four outer folds
    # --------------------------------------------------------

    outer_training = (
        repeat_assignments[
            repeat_assignments["fold"]
            != fold_number
        ]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Create a different deterministic 90/10 split
    # for every repeat/fold
    # --------------------------------------------------------

    inner_split_seed = (
        get_efficientnet_inner_split_seed(
            repeat_number,
            fold_number,
        )
    )

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=0.10,
        random_state=inner_split_seed,
    )

    (
        model_training_positions,
        internal_validation_positions,
    ) = next(
        splitter.split(
            outer_training,
            outer_training["class_index"],
        )
    )


    # --------------------------------------------------------
    # 90% model-training partition
    # --------------------------------------------------------

    model_training = (
        outer_training
        .iloc[model_training_positions]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # 10% internal-validation partition
    # --------------------------------------------------------

    internal_validation = (
        outer_training
        .iloc[internal_validation_positions]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Label the three partitions
    # --------------------------------------------------------

    model_training["partition"] = (
        "model_training"
    )

    internal_validation["partition"] = (
        "internal_validation"
    )

    outer_validation["partition"] = (
        "outer_validation"
    )


    # --------------------------------------------------------
    # Combined manifest
    # --------------------------------------------------------

    partition_manifest = pd.concat(
        [
            model_training,
            internal_validation,
            outer_validation,
        ],
        ignore_index=True,
    )


    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    assert len(outer_training) == 4480

    assert len(model_training) == 4032

    assert len(internal_validation) == 448

    assert len(outer_validation) == 1120

    assert len(partition_manifest) == 5600


    # No image may occur in more than one partition
    assert not (
        partition_manifest[
            "relative_path"
        ]
        .duplicated()
        .any()
    )


    # Exact class balance
    assert (
        model_training[
            "class"
        ]
        .value_counts()
        .eq(1008)
        .all()
    )

    assert (
        internal_validation[
            "class"
        ]
        .value_counts()
        .eq(112)
        .all()
    )

    assert (
        outer_validation[
            "class"
        ]
        .value_counts()
        .eq(280)
        .all()
    )


    return (
        model_training,
        internal_validation,
        outer_validation,
        partition_manifest,
    )


# ============================================================
# TEST ONLY: Repeat 1 / Fold 1
# ============================================================

(
    test_model_training,
    test_internal_validation,
    test_outer_validation,
    test_partition_manifest,
) = make_repeated_efficientnet_partitions(
    repeat_number=1,
    fold_number=1,
)


print("=" * 70)
print("REPEAT 1 / FOLD 1 — 90/10 PARTITION CHECK")
print("=" * 70)

print(
    "Outer-training images:",
    len(test_model_training)
    + len(test_internal_validation),
)

print(
    "Model-training images:",
    len(test_model_training),
)

print(
    "Internal-validation images:",
    len(test_internal_validation),
)

print(
    "Untouched outer-validation images:",
    len(test_outer_validation),
)

print(
    "\nModel-training class counts:"
)

print(
    test_model_training[
        "class"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nInternal-validation class counts:"
)

print(
    test_internal_validation[
        "class"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nOuter-validation class counts:"
)

print(
    test_outer_validation[
        "class"
    ]
    .value_counts()
    .sort_index()
)

print(
    "\nInner split seed:",
    get_efficientnet_inner_split_seed(
        1,
        1,
    ),
)

print(
    "\nStep 8 90/10 partition validation PASSED."
)

REPEAT 1 / FOLD 1 — 90/10 PARTITION CHECK
Outer-training images: 4480
Model-training images: 4032
Internal-validation images: 448
Untouched outer-validation images: 1120

Model-training class counts:
class
glioma        1008
meningioma    1008
notumor       1008
pituitary     1008
Name: count, dtype: int64

Internal-validation class counts:
class
glioma        112
meningioma    112
notumor       112
pituitary     112
Name: count, dtype: int64

Outer-validation class counts:
class
glioma        280
meningioma    280
notumor       280
pituitary     280
Name: count, dtype: int64

Inner split seed: 606101

Step 8 90/10 partition validation PASSED.


In [9]:
# ============================================================
# STEP 9: DEFINE EFFICIENTNETB0 SEARCH AND REPEAT-LEVEL OUTPUTS
# ============================================================

import json
from itertools import product
from pathlib import Path


# ------------------------------------------------------------
# 1. Hyperparameter search space
#
# These 12 configurations will be searched independently
# inside EVERY outer fold using only:
#
#   4,032 model-training images
#     448 internal-validation images
#
# The 1,120 outer-validation images are never used
# to choose these parameters.
# ------------------------------------------------------------

EFFICIENTNETB0_BATCH_SIZES = [
    8,
    16,
    32,
]

EFFICIENTNETB0_HEAD_LEARNING_RATES = [
    1e-3,
    3e-4,
]

EFFICIENTNETB0_FINE_TUNE_LEARNING_RATES = [
    1e-5,
    3e-6,
]


# ------------------------------------------------------------
# 2. Fixed architecture/training settings
#
# These preserve the original EfficientNetB0 experiment.
# ------------------------------------------------------------

EFFICIENTNETB0_DROPOUT_RATE = 0.30

EFFICIENTNETB0_FINE_TUNE_FROM_LAYER = (
    "block7a_expand_conv"
)

EFFICIENTNETB0_HEAD_MAX_EPOCHS = 15

EFFICIENTNETB0_FINE_TUNE_MAX_EPOCHS = 20

EFFICIENTNETB0_EARLY_STOPPING_PATIENCE = 3

EFFICIENTNETB0_HEAD_MIN_LR = 1e-6

EFFICIENTNETB0_FINE_TUNE_MIN_LR = 1e-7


# ------------------------------------------------------------
# 3. Build the 12 candidate configurations
# ------------------------------------------------------------

efficientnetb0_search_configurations = []

configuration_number = 0


for (
    batch_size,
    head_learning_rate,
    fine_tune_learning_rate,
) in product(
    EFFICIENTNETB0_BATCH_SIZES,
    EFFICIENTNETB0_HEAD_LEARNING_RATES,
    EFFICIENTNETB0_FINE_TUNE_LEARNING_RATES,
):

    configuration_number += 1

    configuration_id = (
        f"config_{configuration_number:02d}"
        f"_bs{batch_size}"
        f"_headlr{head_learning_rate:.0e}"
        f"_ftlr{fine_tune_learning_rate:.0e}"
    )

    efficientnetb0_search_configurations.append(
        {
            "configuration_number":
                configuration_number,

            "configuration_id":
                configuration_id,

            "batch_size":
                batch_size,

            "head_learning_rate":
                head_learning_rate,

            "fine_tune_learning_rate":
                fine_tune_learning_rate,
        }
    )


efficientnetb0_search_table = pd.DataFrame(
    efficientnetb0_search_configurations
)


assert len(
    efficientnetb0_search_table
) == 12


# ------------------------------------------------------------
# 4. Configuration-selection rule
#
# Each outer fold will choose ONE winning configuration using
# only its 448-image INTERNAL validation set.
# ------------------------------------------------------------

EFFICIENTNETB0_SELECTION_RULE = [
    "highest internal-validation macro F1",
    "highest internal-validation balanced accuracy",
    "lowest internal-validation log loss",
    "lowest configuration number",
]


# ------------------------------------------------------------
# 5. Persistent output structure
#
# Fold results are saved separately for resume protection.
# After all 5 folds finish, they are aggregated into ONE
# repeat/separation-level result.
# ------------------------------------------------------------

for repeat_number in range(
    1,
    NUMBER_OF_REPEATS + 1,
):

    repeat_directory = (
        REPEATED_EFFICIENTNETB0_DIR
        / f"repeat_{repeat_number:02d}"
    )

    repeat_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# Final table used later for paired statistical testing.
#
# It will eventually contain exactly 10 rows:
# one row for each repeat/separation.
REPEAT_LEVEL_SUMMARY_PATH = (
    REPEATED_EFFICIENTNETB0_DIR
    / "repeat_level_summary.csv"
)


# ------------------------------------------------------------
# 6. Save experiment definition
# ------------------------------------------------------------

SEARCH_DEFINITION_PATH = (
    REPEATED_EFFICIENTNETB0_DIR
    / "repeated_cv_search_definition.json"
)


search_definition = {
    "model":
        "EfficientNetB0",

    "number_of_repeats":
        10,

    "outer_folds_per_repeat":
        5,

    "total_outer_evaluations":
        50,

    "outer_training_images":
        4480,

    "internal_model_training_images":
        4032,

    "internal_validation_images":
        448,

    "outer_validation_images":
        1120,

    "internal_validation_fraction":
        0.10,

    "number_of_candidate_configurations":
        12,

    "batch_sizes":
        EFFICIENTNETB0_BATCH_SIZES,

    "head_learning_rates":
        EFFICIENTNETB0_HEAD_LEARNING_RATES,

    "fine_tune_learning_rates":
        EFFICIENTNETB0_FINE_TUNE_LEARNING_RATES,

    "dropout_rate":
        EFFICIENTNETB0_DROPOUT_RATE,

    "fine_tune_from_layer":
        EFFICIENTNETB0_FINE_TUNE_FROM_LAYER,

    "head_max_epochs":
        EFFICIENTNETB0_HEAD_MAX_EPOCHS,

    "fine_tune_max_epochs":
        EFFICIENTNETB0_FINE_TUNE_MAX_EPOCHS,

    "early_stopping_patience":
        EFFICIENTNETB0_EARLY_STOPPING_PATIENCE,

    "selection_rule":
        EFFICIENTNETB0_SELECTION_RULE,

    "preprocessing":
        (
            "grayscale to pseudo-RGB; "
            "float32 0-255 input; "
            "EfficientNetB0 internal rescaling"
        ),

    "final_statistical_unit":
        (
            "mean of 5 outer-fold results "
            "within each repeat"
        ),

    "expected_repeat_level_results":
        10,

    "testing_images_used":
        False,
}


temporary_definition_path = (
    SEARCH_DEFINITION_PATH
    .with_name(
        SEARCH_DEFINITION_PATH.name
        + ".tmp"
    )
)


with temporary_definition_path.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        search_definition,
        file,
        indent=4,
    )


temporary_definition_path.replace(
    SEARCH_DEFINITION_PATH
)


# ------------------------------------------------------------
# 7. Display
# ------------------------------------------------------------

print("=" * 70)
print("EFFICIENTNETB0 REPEATED-CV SEARCH DEFINITION")
print("=" * 70)

print(
    "Candidate configurations:",
    len(
        efficientnetb0_search_configurations
    ),
)

display(
    efficientnetb0_search_table
)

print(
    "\nFine-tuning begins from:",
    EFFICIENTNETB0_FINE_TUNE_FROM_LAYER,
)

print(
    "\nOuter folds per repeat:",
    NUMBER_OF_OUTER_FOLDS,
)

print(
    "Repeats/separations:",
    NUMBER_OF_REPEATS,
)

print(
    "Total outer evaluations:",
    NUMBER_OF_REPEATS
    * NUMBER_OF_OUTER_FOLDS,
)

print(
    "\nParameters are re-selected "
    "inside EVERY outer fold."
)

print(
    "\nFold-level results will be saved "
    "for checkpoint/resume protection."
)

print(
    "Five folds will then be averaged "
    "into ONE result per repeat."
)

print(
    "\nFinal Wilcoxon table will contain:",
    10,
    "EfficientNetB0 values.",
)

print(
    "\nRepeat-level summary path:"
)

print(
    REPEAT_LEVEL_SUMMARY_PATH
)

print(
    "\nStep 9 search/output definition PASSED."
)

EFFICIENTNETB0 REPEATED-CV SEARCH DEFINITION
Candidate configurations: 12


,configuration_number,configuration_id,batch_size,head_learning_rate,fine_tune_learning_rate
0,1,config_01_bs8_headlr1e-03_ftlr1e-05,8,0.0010,0.000010
1,2,config_02_bs8_headlr1e-03_ftlr3e-06,8,0.0010,0.000003
2,3,config_03_bs8_headlr3e-04_ftlr1e-05,8,0.0003,0.000010
3,4,config_04_bs8_headlr3e-04_ftlr3e-06,8,0.0003,0.000003
4,5,config_05_bs16_headlr1e-03_ftlr1e-05,16,0.0010,0.000010
5,6,config_06_bs16_headlr1e-03_ftlr3e-06,16,0.0010,0.000003
6,7,config_07_bs16_headlr3e-04_ftlr1e-05,16,0.0003,0.000010
7,8,config_08_bs16_headlr3e-04_ftlr3e-06,16,0.0003,0.000003
8,9,config_09_bs32_headlr1e-03_ftlr1e-05,32,0.0010,0.000010
9,10,config_10_bs32_headlr1e-03_ftlr3e-06,32,0.0010,0.000003



Fine-tuning begins from: block7a_expand_conv

Outer folds per repeat: 5
Repeats/separations: 10
Total outer evaluations: 50

Parameters are re-selected inside EVERY outer fold.

Fold-level results will be saved for checkpoint/resume protection.
Five folds will then be averaged into ONE result per repeat.

Final Wilcoxon table will contain: 10 EfficientNetB0 values.

Repeat-level summary path:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0/repeat_level_summary.csv

Step 9 search/output definition PASSED.


In [10]:
# ============================================================
# STEP 10: DEFINE EFFICIENTNETB0 MODEL AND TRAINING UTILITIES
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    log_loss,
    precision_recall_fscore_support,
)


# ------------------------------------------------------------
# 1. Build a fresh ImageNet-pretrained EfficientNetB0
# ------------------------------------------------------------

def build_repeated_efficientnetb0_model():
    """
    Build a fresh EfficientNetB0 transfer-learning model.

    Stage 1:
        - ImageNet backbone frozen
        - new classification head trained

    Stage 2:
        - upper EfficientNetB0 layers unfrozen
        - Batch Normalization layers remain frozen

    IMPORTANT:
        EfficientNetB0 contains its own input rescaling.
        Input tensors therefore remain float32 in the
        original 0-255 range.
    """

    inputs = tf.keras.Input(
        shape=(
            IMAGE_HEIGHT,
            IMAGE_WIDTH,
            IMAGE_CHANNELS,
        ),
        name="input_image",
    )

    base_model = (
        tf.keras.applications.EfficientNetB0(
            include_top=False,
            weights="imagenet",
            input_shape=(
                IMAGE_HEIGHT,
                IMAGE_WIDTH,
                IMAGE_CHANNELS,
            ),
            pooling="avg",
        )
    )

    # Stage 1 starts with complete backbone frozen
    base_model.trainable = False

    # Preserve pretrained BatchNorm behaviour
    # exactly as in the original EfficientNetB0 model.
    features = base_model(
        inputs,
        training=False,
    )

    features = tf.keras.layers.Dropout(
        rate=EFFICIENTNETB0_DROPOUT_RATE,
        name="classifier_dropout",
    )(
        features
    )

    outputs = tf.keras.layers.Dense(
        units=NUMBER_OF_CLASSES,
        activation="softmax",
        dtype="float32",
        name="class_probabilities",
    )(
        features
    )

    model = tf.keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="efficientnetb0_repeated_cv",
    )

    return model, base_model


# ------------------------------------------------------------
# 2. Compile model
# ------------------------------------------------------------

def compile_repeated_efficientnetb0(
    model,
    learning_rate,
):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss=(
            tf.keras.losses
            .SparseCategoricalCrossentropy()
        ),
        metrics=[
            tf.keras.metrics
            .SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],
    )


# ------------------------------------------------------------
# 3. Enable Stage-2 fine-tuning
# ------------------------------------------------------------

def enable_repeated_efficientnetb0_fine_tuning(
    base_model,
):
    """
    Unfreeze layers beginning at block7a_expand_conv.

    Batch Normalization layers remain frozen.
    """

    base_model.trainable = True

    start_found = False

    for layer in base_model.layers:

        if (
            layer.name
            == EFFICIENTNETB0_FINE_TUNE_FROM_LAYER
        ):
            start_found = True

        layer.trainable = (
            start_found
            and not isinstance(
                layer,
                tf.keras.layers.BatchNormalization,
            )
        )

    if not start_found:
        raise ValueError(
            "Fine-tuning start layer "
            "was not found: "
            f"{EFFICIENTNETB0_FINE_TUNE_FROM_LAYER}"
        )

    trainable_layers = [
        layer.name
        for layer in base_model.layers
        if layer.trainable
    ]

    if not trainable_layers:
        raise RuntimeError(
            "No EfficientNetB0 backbone layers "
            "were enabled for fine-tuning."
        )

    return trainable_layers


# ------------------------------------------------------------
# 4. Training callbacks
# ------------------------------------------------------------

def create_repeated_efficientnetb0_callbacks(
    checkpoint_path,
    minimum_learning_rate,
):
    """
    All checkpoint/early-stopping decisions use ONLY the
    448-image internal-validation partition.
    """

    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(
                checkpoint_path
            ),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=True,
            verbose=0,
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=(
                EFFICIENTNETB0_EARLY_STOPPING_PATIENCE
            ),
            restore_best_weights=True,
            verbose=0,
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.2,
            patience=2,
            min_lr=minimum_learning_rate,
            verbose=0,
        ),

        tf.keras.callbacks.TerminateOnNaN(),
    ]


# ------------------------------------------------------------
# 5. Find best epoch and validation loss
# ------------------------------------------------------------

def get_repeated_efficientnetb0_best_epoch(
    history,
):
    validation_losses = np.asarray(
        history.history["val_loss"],
        dtype=np.float64,
    )

    if validation_losses.size == 0:
        raise RuntimeError(
            "No validation losses were recorded."
        )

    if not np.isfinite(
        validation_losses
    ).all():
        raise RuntimeError(
            "Non-finite validation loss detected."
        )

    best_position = int(
        np.argmin(
            validation_losses
        )
    )

    return {
        "best_epoch":
            best_position + 1,

        "best_validation_loss":
            float(
                validation_losses[
                    best_position
                ]
            ),
    }


# ------------------------------------------------------------
# 6. Classification metrics
# ------------------------------------------------------------

def calculate_repeated_efficientnetb0_metrics(
    true_labels,
    predicted_labels,
    probabilities,
):
    """
    Used for both internal-validation model selection
    and final untouched outer-fold evaluation.
    """

    macro_scores = (
        precision_recall_fscore_support(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        )
    )

    return {
        "accuracy":
            float(
                accuracy_score(
                    true_labels,
                    predicted_labels,
                )
            ),

        "balanced_accuracy":
            float(
                balanced_accuracy_score(
                    true_labels,
                    predicted_labels,
                )
            ),

        "macro_precision":
            float(
                macro_scores[0]
            ),

        "macro_recall":
            float(
                macro_scores[1]
            ),

        "macro_f1":
            float(
                macro_scores[2]
            ),

        "log_loss":
            float(
                log_loss(
                    true_labels,
                    probabilities,
                    labels=list(
                        range(
                            NUMBER_OF_CLASSES
                        )
                    ),
                )
            ),
    }


# ------------------------------------------------------------
# 7. Basic model-construction test
# ------------------------------------------------------------

tf.keras.backend.clear_session()

test_model, test_base_model = (
    build_repeated_efficientnetb0_model()
)


assert (
    test_model.output_shape
    == (None, 4)
)

assert (
    test_base_model.trainable
    is False
)

assert (
    EFFICIENTNETB0_FINE_TUNE_FROM_LAYER
    in [
        layer.name
        for layer
        in test_base_model.layers
    ]
)


print("=" * 70)
print("EFFICIENTNETB0 MODEL UTILITY CHECK")
print("=" * 70)

print(
    "Model output shape:",
    test_model.output_shape,
)

print(
    "Backbone initially trainable:",
    test_base_model.trainable,
)

print(
    "Dropout rate:",
    EFFICIENTNETB0_DROPOUT_RATE,
)

print(
    "Fine-tuning starts from:",
    EFFICIENTNETB0_FINE_TUNE_FROM_LAYER,
)

print(
    "Fine-tuning start layer exists: True"
)

print(
    "\nStep 10 model/training utilities PASSED."
)


# Release the temporary test model
del test_model
del test_base_model

tf.keras.backend.clear_session()
gc.collect()

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
EFFICIENTNETB0 MODEL UTILITY CHECK
Model output shape: (None, 4)
Backbone initially trainable: False
Dropout rate: 0.3
Fine-tuning starts from: block7a_expand_conv
Fine-tuning start layer exists: True

Step 10 model/training utilities PASSED.


0

In [11]:
# ============================================================
# STEP 11: DEFINE EFFICIENTNETB0 CANDIDATE SAVE / RESUME UTILITIES
# ============================================================

import json
from pathlib import Path


# ------------------------------------------------------------
# Atomic JSON save
# ------------------------------------------------------------

def save_json_atomic(
    data,
    destination,
):
    destination = Path(
        destination
    )

    temporary_path = (
        destination.with_name(
            destination.name + ".tmp"
        )
    )

    with temporary_path.open(
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            data,
            file,
            indent=4,
            default=str,
        )

    temporary_path.replace(
        destination
    )


# ------------------------------------------------------------
# Atomic CSV save
# ------------------------------------------------------------

def save_dataframe_atomic(
    dataframe,
    destination,
    index=False,
):
    destination = Path(
        destination
    )

    temporary_path = (
        destination.with_name(
            destination.name + ".tmp"
        )
    )

    dataframe.to_csv(
        temporary_path,
        index=index,
    )

    temporary_path.replace(
        destination
    )


# ------------------------------------------------------------
# Candidate output directory
# ------------------------------------------------------------

def get_efficientnetb0_candidate_directory(
    repeat_number,
    fold_number,
    configuration_id,
):
    return (
        REPEATED_EFFICIENTNETB0_DIR
        / f"repeat_{repeat_number:02d}"
        / f"fold_{fold_number:02d}"
        / "configurations"
        / configuration_id
    )


# ------------------------------------------------------------
# Check whether one candidate completed successfully
# ------------------------------------------------------------

def validate_completed_efficientnetb0_candidate(
    repeat_number,
    fold_number,
    configuration_id,
):
    candidate_directory = (
        get_efficientnetb0_candidate_directory(
            repeat_number,
            fold_number,
            configuration_id,
        )
    )

    configuration_path = (
        candidate_directory
        / "configuration.json"
    )

    metrics_path = (
        candidate_directory
        / "internal_validation_metrics.json"
    )

    history_path = (
        candidate_directory
        / "training_history.csv"
    )

    completion_path = (
        candidate_directory
        / "complete.json"
    )

    required_paths = [
        configuration_path,
        metrics_path,
        history_path,
        completion_path,
    ]

    if not all(
        path.exists()
        for path in required_paths
    ):
        return False

    try:

        with completion_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            completion = json.load(
                file
            )

        if (
            completion.get("status")
            != "completed"
        ):
            return False

        if (
            int(
                completion["repeat"]
            )
            != repeat_number
        ):
            return False

        if (
            int(
                completion["fold"]
            )
            != fold_number
        ):
            return False

        if (
            completion[
                "configuration_id"
            ]
            != configuration_id
        ):
            return False

        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            metrics = json.load(
                file
            )

        history = pd.read_csv(
            history_path
        )

        if history.empty:
            return False

        if (
            not np.isfinite(
                float(
                    metrics[
                        "internal_macro_f1"
                    ]
                )
            )
        ):
            return False

        return True

    except Exception:

        return False


# ------------------------------------------------------------
# Test directory construction
# ------------------------------------------------------------

test_candidate_directory = (
    get_efficientnetb0_candidate_directory(
        repeat_number=1,
        fold_number=1,
        configuration_id=(
            efficientnetb0_search_configurations[
                0
            ][
                "configuration_id"
            ]
        ),
    )
)


print("=" * 70)
print("EFFICIENTNETB0 SAVE / RESUME STRUCTURE")
print("=" * 70)

print(
    "Example candidate directory:"
)

print(
    test_candidate_directory
)

print(
    "\nCandidate completion currently:",
    validate_completed_efficientnetb0_candidate(
        repeat_number=1,
        fold_number=1,
        configuration_id=(
            efficientnetb0_search_configurations[
                0
            ][
                "configuration_id"
            ]
        ),
    ),
)

print(
    "\nEach configuration will save:"
)

print(
    "  configuration.json"
)

print(
    "  internal_validation_metrics.json"
)

print(
    "  training_history.csv"
)

print(
    "  complete.json"
)

print(
    "\nStep 11 candidate persistence setup PASSED."
)

EFFICIENTNETB0 SAVE / RESUME STRUCTURE
Example candidate directory:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0/repeat_01/fold_01/configurations/config_01_bs8_headlr1e-03_ftlr1e-05

Candidate completion currently: True

Each configuration will save:
  configuration.json
  internal_validation_metrics.json
  training_history.csv
  complete.json

Step 11 candidate persistence setup PASSED.


In [12]:
# ============================================================
# STEP 13: FULL EFFICIENTNETB0 REPEATED NESTED-CV EXPERIMENT
#
# 10 repeats × 5 outer folds = 50 outer evaluations
#
# For EACH outer fold:
#
#   4,480 outer-training images
#       ↓
#   4,032 model training
#     448 internal validation
#       ↓
#   search all 12 EfficientNetB0 configurations
#       ↓
#   select winner using INTERNAL validation only
#       ↓
#   rebuild fresh EfficientNetB0
#       ↓
#   refit winner on ALL 4,480 outer-training images
#       ↓
#   evaluate ONCE on untouched 1,120 outer-validation images
#
# After 5 folds:
#   aggregate into ONE repeat/separation result.
#
# Final statistical table:
#   10 rows = 10 repeats/separations
# ============================================================

import gc
import json
import random
import shutil
import time

from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
)


# ============================================================
# 1. ADDITIONAL REPRODUCIBLE REFIT SEED
# ============================================================

# Candidate model/training seeds use the 707000 schedule
# defined earlier.
#
# Final refitting receives a separate deterministic seed
# schedule so that the final 4,480-image refit is a fresh
# training procedure rather than a continuation of the
# candidate-selection training.

BASE_REFIT_SEED = 808000


def get_efficientnet_refit_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_REFIT_SEED
        + repeat_number * 100
        + fold_number
    )


# ============================================================
# 2. FINAL FOLD OUTPUT VALIDATION
# ============================================================

def validate_completed_efficientnetb0_fold(
    repeat_number,
    fold_number,
):
    """
    A fold counts as completed only when all required
    final outputs exist and pass basic validation.
    """

    fold_directory = (
        REPEATED_EFFICIENTNETB0_DIR
        / f"repeat_{repeat_number:02d}"
        / f"fold_{fold_number:02d}"
    )

    metrics_path = (
        fold_directory
        / "outer_fold_metrics.json"
    )

    selected_configuration_path = (
        fold_directory
        / "selected_configuration.json"
    )

    predictions_path = (
        fold_directory
        / "outer_validation_predictions.csv"
    )

    confusion_matrix_path = (
        fold_directory
        / "confusion_matrix.csv"
    )

    classification_report_path = (
        fold_directory
        / "classification_report.csv"
    )

    partition_manifest_path = (
        fold_directory
        / "partition_manifest.csv"
    )

    candidate_leaderboard_path = (
        fold_directory
        / "candidate_leaderboard.csv"
    )

    final_history_path = (
        fold_directory
        / "final_refit_history.csv"
    )

    final_weights_path = (
        fold_directory
        / "final_selected_model.weights.h5"
    )

    completion_path = (
        fold_directory
        / "fold_complete.json"
    )

    required_paths = [
        metrics_path,
        selected_configuration_path,
        predictions_path,
        confusion_matrix_path,
        classification_report_path,
        partition_manifest_path,
        candidate_leaderboard_path,
        final_history_path,
        final_weights_path,
        completion_path,
    ]

    if not all(
        path.exists()
        for path in required_paths
    ):
        return False

    try:

        with completion_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            completion = json.load(
                file
            )

        if (
            completion.get("status")
            != "completed"
        ):
            return False

        if (
            int(
                completion["repeat"]
            )
            != repeat_number
        ):
            return False

        if (
            int(
                completion["fold"]
            )
            != fold_number
        ):
            return False

        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            metrics = json.load(
                file
            )

        if (
            int(
                metrics["repeat"]
            )
            != repeat_number
        ):
            return False

        if (
            int(
                metrics["fold"]
            )
            != fold_number
        ):
            return False

        predictions = pd.read_csv(
            predictions_path
        )

        if len(predictions) != 1120:
            return False

        if (
            predictions[
                "relative_path"
            ]
            .nunique()
            != 1120
        ):
            return False

        leaderboard = pd.read_csv(
            candidate_leaderboard_path
        )

        if len(leaderboard) != 12:
            return False

        history = pd.read_csv(
            final_history_path
        )

        if history.empty:
            return False

        return True

    except Exception:

        return False


# ============================================================
# 3. RUN ONE INNER EFFICIENTNETB0 CANDIDATE
# ============================================================

def run_efficientnetb0_candidate(
    repeat_number,
    fold_number,
    configuration,
    model_training_dataframe,
    internal_validation_dataframe,
):
    """
    Train one EfficientNetB0 candidate using only the
    4,032 / 448 internal train/validation split.

    The 1,120-image outer-validation partition is NEVER
    used inside this function.
    """

    configuration_id = (
        configuration[
            "configuration_id"
        ]
    )

    configuration_number = int(
        configuration[
            "configuration_number"
        ]
    )

    batch_size = int(
        configuration[
            "batch_size"
        ]
    )

    head_learning_rate = float(
        configuration[
            "head_learning_rate"
        ]
    )

    fine_tune_learning_rate = float(
        configuration[
            "fine_tune_learning_rate"
        ]
    )


    # --------------------------------------------------------
    # Candidate directory
    # --------------------------------------------------------

    candidate_directory = (
        get_efficientnetb0_candidate_directory(
            repeat_number,
            fold_number,
            configuration_id,
        )
    )


    # --------------------------------------------------------
    # Resume completed candidate
    # --------------------------------------------------------

    if validate_completed_efficientnetb0_candidate(
        repeat_number,
        fold_number,
        configuration_id,
    ):

        print(
            f"  {configuration_id}: "
            "already completed — skipping."
        )

        metrics_path = (
            candidate_directory
            / "internal_validation_metrics.json"
        )

        with metrics_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            return json.load(
                file
            )


    # --------------------------------------------------------
    # Delete incomplete candidate only
    # --------------------------------------------------------

    if candidate_directory.exists():

        print(
            f"  {configuration_id}: "
            "incomplete output found — rerunning."
        )

        shutil.rmtree(
            candidate_directory
        )


    candidate_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    configuration_path = (
        candidate_directory
        / "configuration.json"
    )

    metrics_path = (
        candidate_directory
        / "internal_validation_metrics.json"
    )

    history_path = (
        candidate_directory
        / "training_history.csv"
    )

    completion_path = (
        candidate_directory
        / "complete.json"
    )

    stage_1_checkpoint = (
        candidate_directory
        / "stage_1_best.weights.h5"
    )

    stage_2_checkpoint = (
        candidate_directory
        / "stage_2_best.weights.h5"
    )


    # --------------------------------------------------------
    # Same deterministic model seed for all candidates
    # within the same outer fold.
    #
    # This keeps initialization/training randomness controlled
    # when comparing the 12 hyperparameter configurations.
    # --------------------------------------------------------

    candidate_seed = (
        get_efficientnet_model_seed(
            repeat_number,
            fold_number,
        )
    )


    tf.keras.backend.clear_session()

    random.seed(
        candidate_seed
    )

    np.random.seed(
        candidate_seed
    )

    tf.keras.utils.set_random_seed(
        candidate_seed
    )

    gc.collect()


    # --------------------------------------------------------
    # Datasets
    # --------------------------------------------------------

    training_dataset = (
        make_efficientnetb0_dataset(
            model_training_dataframe,
            training=True,
            seed=candidate_seed,
            batch_size=batch_size,
        )
    )

    internal_validation_dataset = (
        make_efficientnetb0_dataset(
            internal_validation_dataframe,
            training=False,
            seed=candidate_seed,
            batch_size=batch_size,
        )
    )


    # --------------------------------------------------------
    # Fresh EfficientNetB0
    # --------------------------------------------------------

    model, base_model = (
        build_repeated_efficientnetb0_model()
    )


    candidate_start = (
        time.perf_counter()
    )


    # ========================================================
    # STAGE 1 — FROZEN BACKBONE
    # ========================================================

    compile_repeated_efficientnetb0(
        model,
        head_learning_rate,
    )


    stage_1_start = (
        time.perf_counter()
    )


    stage_1_history = model.fit(
        training_dataset,

        validation_data=(
            internal_validation_dataset
        ),

        epochs=(
            EFFICIENTNETB0_HEAD_MAX_EPOCHS
        ),

        callbacks=(
            create_repeated_efficientnetb0_callbacks(
                stage_1_checkpoint,
                EFFICIENTNETB0_HEAD_MIN_LR,
            )
        ),

        verbose=0,
    )


    stage_1_seconds = (
        time.perf_counter()
        - stage_1_start
    )


    stage_1_best = (
        get_repeated_efficientnetb0_best_epoch(
            stage_1_history
        )
    )


    if not stage_1_checkpoint.exists():

        raise RuntimeError(
            "Stage 1 checkpoint was not saved."
        )


    # Fine-tuning must begin from the best Stage-1 weights.
    model.load_weights(
        stage_1_checkpoint
    )


    # ========================================================
    # STAGE 2 — FINE-TUNING
    # ========================================================

    trainable_layers = (
        enable_repeated_efficientnetb0_fine_tuning(
            base_model
        )
    )


    compile_repeated_efficientnetb0(
        model,
        fine_tune_learning_rate,
    )


    stage_2_start = (
        time.perf_counter()
    )


    stage_2_history = model.fit(
        training_dataset,

        validation_data=(
            internal_validation_dataset
        ),

        epochs=(
            EFFICIENTNETB0_FINE_TUNE_MAX_EPOCHS
        ),

        callbacks=(
            create_repeated_efficientnetb0_callbacks(
                stage_2_checkpoint,
                EFFICIENTNETB0_FINE_TUNE_MIN_LR,
            )
        ),

        verbose=0,
    )


    stage_2_seconds = (
        time.perf_counter()
        - stage_2_start
    )


    stage_2_best = (
        get_repeated_efficientnetb0_best_epoch(
            stage_2_history
        )
    )


    if not stage_2_checkpoint.exists():

        raise RuntimeError(
            "Stage 2 checkpoint was not saved."
        )


    # ========================================================
    # SELECT STAGE USING INTERNAL VALIDATION ONLY
    # ========================================================

    if (
        stage_2_best[
            "best_validation_loss"
        ]
        <
        stage_1_best[
            "best_validation_loss"
        ]
    ):

        selected_stage = (
            "fine_tuning"
        )

        selected_checkpoint = (
            stage_2_checkpoint
        )

        selected_epoch = int(
            stage_2_best[
                "best_epoch"
            ]
        )

        selected_validation_loss = float(
            stage_2_best[
                "best_validation_loss"
            ]
        )

    else:

        selected_stage = (
            "frozen_head"
        )

        selected_checkpoint = (
            stage_1_checkpoint
        )

        selected_epoch = int(
            stage_1_best[
                "best_epoch"
            ]
        )

        selected_validation_loss = float(
            stage_1_best[
                "best_validation_loss"
            ]
        )


    model.load_weights(
        selected_checkpoint
    )


    # ========================================================
    # INTERNAL VALIDATION PREDICTIONS
    # ========================================================

    internal_probabilities = (
        model.predict(
            internal_validation_dataset,
            verbose=0,
        )
    )


    internal_probabilities = np.asarray(
        internal_probabilities,
        dtype=np.float32,
    )


    if (
        internal_probabilities.shape
        != (
            448,
            NUMBER_OF_CLASSES,
        )
    ):

        raise RuntimeError(
            "Unexpected internal-validation "
            "prediction shape."
        )


    if not np.isfinite(
        internal_probabilities
    ).all():

        raise RuntimeError(
            "Non-finite internal-validation "
            "probabilities detected."
        )


    if not np.allclose(
        internal_probabilities.sum(
            axis=1
        ),
        1.0,
        atol=1e-5,
    ):

        raise RuntimeError(
            "Internal probability rows "
            "do not sum to one."
        )


    internal_true_labels = (
        internal_validation_dataframe[
            "class_index"
        ]
        .to_numpy(
            dtype=np.int32
        )
    )


    internal_predicted_labels = (
        np.argmax(
            internal_probabilities,
            axis=1,
        )
        .astype(np.int32)
    )


    internal_metrics = (
        calculate_repeated_efficientnetb0_metrics(
            internal_true_labels,
            internal_predicted_labels,
            internal_probabilities,
        )
    )


    candidate_total_seconds = (
        time.perf_counter()
        - candidate_start
    )


    # ========================================================
    # SAVE TRAINING HISTORY
    # ========================================================

    stage_1_history_df = pd.DataFrame(
        stage_1_history.history
    )

    stage_1_history_df.insert(
        0,
        "stage_epoch",
        np.arange(
            1,
            len(stage_1_history_df) + 1,
        ),
    )

    stage_1_history_df.insert(
        1,
        "stage",
        "frozen_head",
    )


    stage_2_history_df = pd.DataFrame(
        stage_2_history.history
    )

    stage_2_history_df.insert(
        0,
        "stage_epoch",
        np.arange(
            1,
            len(stage_2_history_df) + 1,
        ),
    )

    stage_2_history_df.insert(
        1,
        "stage",
        "fine_tuning",
    )


    combined_history_df = pd.concat(
        [
            stage_1_history_df,
            stage_2_history_df,
        ],
        ignore_index=True,
    )


    # ========================================================
    # CANDIDATE RECORD
    # ========================================================

    candidate_configuration = {
        "model":
            "EfficientNetB0",

        "repeat":
            int(repeat_number),

        "fold":
            int(fold_number),

        "configuration_number":
            configuration_number,

        "configuration_id":
            configuration_id,

        "batch_size":
            batch_size,

        "head_learning_rate":
            head_learning_rate,

        "fine_tune_learning_rate":
            fine_tune_learning_rate,

        "model_seed":
            int(candidate_seed),

        "inner_split_seed":
            int(
                get_efficientnet_inner_split_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "model_training_images":
            4032,

        "internal_validation_images":
            448,

        "outer_validation_used_for_selection":
            False,
    }


    candidate_metrics = {
        "model":
            "EfficientNetB0",

        "repeat":
            int(repeat_number),

        "fold":
            int(fold_number),

        "configuration_number":
            configuration_number,

        "configuration_id":
            configuration_id,

        "batch_size":
            batch_size,

        "head_learning_rate":
            head_learning_rate,

        "fine_tune_learning_rate":
            fine_tune_learning_rate,

        "selected_stage":
            selected_stage,

        "selected_epoch":
            selected_epoch,

        "selected_validation_loss":
            selected_validation_loss,

        "stage_1_best_epoch":
            int(
                stage_1_best[
                    "best_epoch"
                ]
            ),

        "stage_1_best_validation_loss":
            float(
                stage_1_best[
                    "best_validation_loss"
                ]
            ),

        "stage_2_best_epoch":
            int(
                stage_2_best[
                    "best_epoch"
                ]
            ),

        "stage_2_best_validation_loss":
            float(
                stage_2_best[
                    "best_validation_loss"
                ]
            ),

        "internal_accuracy":
            float(
                internal_metrics[
                    "accuracy"
                ]
            ),

        "internal_balanced_accuracy":
            float(
                internal_metrics[
                    "balanced_accuracy"
                ]
            ),

        "internal_macro_precision":
            float(
                internal_metrics[
                    "macro_precision"
                ]
            ),

        "internal_macro_recall":
            float(
                internal_metrics[
                    "macro_recall"
                ]
            ),

        "internal_macro_f1":
            float(
                internal_metrics[
                    "macro_f1"
                ]
            ),

        "internal_log_loss":
            float(
                internal_metrics[
                    "log_loss"
                ]
            ),

        "stage_1_training_seconds":
            float(
                stage_1_seconds
            ),

        "stage_2_training_seconds":
            float(
                stage_2_seconds
            ),

        "total_candidate_seconds":
            float(
                candidate_total_seconds
            ),

        "trainable_backbone_layers":
            len(
                trainable_layers
            ),
    }


    # ========================================================
    # SAVE CANDIDATE
    # ========================================================

    save_dataframe_atomic(
        combined_history_df,
        history_path,
        index=False,
    )

    save_json_atomic(
        candidate_configuration,
        configuration_path,
    )

    save_json_atomic(
        candidate_metrics,
        metrics_path,
    )


    # Completion marker written LAST.
    completion_information = {
        "status":
            "completed",

        "repeat":
            int(repeat_number),

        "fold":
            int(fold_number),

        "configuration_id":
            configuration_id,

        "internal_macro_f1":
            float(
                internal_metrics[
                    "macro_f1"
                ]
            ),

        "outer_validation_used":
            False,

        "completed_at":
            datetime.now().isoformat(),
    }


    save_json_atomic(
        completion_information,
        completion_path,
    )


    if not validate_completed_efficientnetb0_candidate(
        repeat_number,
        fold_number,
        configuration_id,
    ):

        raise RuntimeError(
            "Candidate persistence validation failed."
        )


    # Candidate checkpoints are no longer required once the
    # metrics/history/completion files have been validated.
    for checkpoint_path in [
        stage_1_checkpoint,
        stage_2_checkpoint,
    ]:

        if checkpoint_path.exists():

            checkpoint_path.unlink()


    print(
        f"  {configuration_id}: "
        f"macro F1="
        f"{internal_metrics['macro_f1']:.4f} "
        f"stage={selected_stage}"
    )


    # Cleanup
    del model
    del base_model
    del training_dataset
    del internal_validation_dataset
    del internal_probabilities
    del stage_1_history
    del stage_2_history

    tf.keras.backend.clear_session()

    gc.collect()


    return candidate_metrics


# ============================================================
# 4. BUILD / UPDATE GLOBAL 10-REPEAT SUMMARY
# ============================================================

def update_efficientnetb0_repeat_level_summary():
    """
    Reconstruct the global statistical table from all completed
    repeat_summary.json files.

    This makes the summary safe to rebuild after interruption.
    """

    records = []


    for repeat_number in range(
        1,
        NUMBER_OF_REPEATS + 1,
    ):

        repeat_summary_path = (
            REPEATED_EFFICIENTNETB0_DIR
            / f"repeat_{repeat_number:02d}"
            / "repeat_summary.json"
        )


        if not repeat_summary_path.exists():

            continue


        try:

            with repeat_summary_path.open(
                "r",
                encoding="utf-8",
            ) as file:

                record = json.load(
                    file
                )

            if (
                record.get("status")
                == "completed"
            ):

                records.append(
                    record
                )

        except Exception:

            continue


    if not records:

        return


    repeat_summary_df = pd.DataFrame(
        records
    )


    repeat_summary_df = (
        repeat_summary_df
        .sort_values(
            "repeat"
        )
        .reset_index(
            drop=True
        )
    )


    save_dataframe_atomic(
        repeat_summary_df,
        REPEAT_LEVEL_SUMMARY_PATH,
        index=False,
    )


# ============================================================
# 5. MAIN 10 × 5 LOOP
# ============================================================

for repeat_number in range(
    1,
    NUMBER_OF_REPEATS + 1,
):

    print(
        "\n"
        + "#" * 80
    )

    print(
        f"EFFICIENTNETB0 — REPEAT / SEPARATION "
        f"{repeat_number:02d} OF "
        f"{NUMBER_OF_REPEATS}"
    )

    print(
        "#" * 80
    )


    repeat_directory = (
        REPEATED_EFFICIENTNETB0_DIR
        / f"repeat_{repeat_number:02d}"
    )


    repeat_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # FIVE OUTER FOLDS
    # ========================================================

    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    ):

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"REPEAT {repeat_number:02d} "
            f"/ FOLD {fold_number:02d}"
        )

        print(
            "=" * 80
        )


        fold_directory = (
            repeat_directory
            / f"fold_{fold_number:02d}"
        )


        fold_directory.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ----------------------------------------------------
        # Completed fold -> skip
        # ----------------------------------------------------

        if validate_completed_efficientnetb0_fold(
            repeat_number,
            fold_number,
        ):

            print(
                "Valid completed fold found — skipping."
            )

            continue


        # ----------------------------------------------------
        # Create 4,032 / 448 / 1,120 partitions
        # ----------------------------------------------------

        (
            model_training_dataframe,
            internal_validation_dataframe,
            outer_validation_dataframe,
            partition_manifest,
        ) = make_repeated_efficientnet_partitions(
            repeat_number,
            fold_number,
        )


        # Complete 4,480-image outer-training partition.
        outer_training_dataframe = (
            pd.concat(
                [
                    model_training_dataframe,
                    internal_validation_dataframe,
                ],
                ignore_index=True,
            )
        )


        assert (
            len(
                outer_training_dataframe
            )
            == 4480
        )

        assert (
            outer_training_dataframe[
                "relative_path"
            ]
            .nunique()
            == 4480
        )


        # ----------------------------------------------------
        # Save exact partition manifest
        # ----------------------------------------------------

        partition_manifest_path = (
            fold_directory
            / "partition_manifest.csv"
        )


        manifest_columns = [
            "relative_path",
            "class",
            "class_index",
            "repeat",
            "fold",
            "split_seed",
            "partition",
        ]


        save_dataframe_atomic(
            partition_manifest[
                manifest_columns
            ],
            partition_manifest_path,
            index=False,
        )


        print(
            "Model training:",
            len(
                model_training_dataframe
            ),
        )

        print(
            "Internal validation:",
            len(
                internal_validation_dataframe
            ),
        )

        print(
            "Untouched outer validation:",
            len(
                outer_validation_dataframe
            ),
        )


        # ====================================================
        # SEARCH ALL 12 CONFIGURATIONS
        # ====================================================

        print(
            "\nSearching 12 configurations..."
        )


        candidate_records = []


        for configuration in (
            efficientnetb0_search_configurations
        ):

            candidate_metrics = (
                run_efficientnetb0_candidate(
                    repeat_number,
                    fold_number,
                    configuration,
                    model_training_dataframe,
                    internal_validation_dataframe,
                )
            )

            candidate_records.append(
                candidate_metrics
            )


        # ====================================================
        # CANDIDATE LEADERBOARD
        # ====================================================

        candidate_leaderboard = pd.DataFrame(
            candidate_records
        )


        if len(
            candidate_leaderboard
        ) != 12:

            raise RuntimeError(
                "Expected 12 completed candidate results."
            )


        # Predetermined selection:
        #
        # 1. highest internal macro F1
        # 2. highest internal balanced accuracy
        # 3. lowest internal log loss
        # 4. lowest configuration number

        candidate_leaderboard = (
            candidate_leaderboard
            .sort_values(
                by=[
                    "internal_macro_f1",
                    "internal_balanced_accuracy",
                    "internal_log_loss",
                    "configuration_number",
                ],

                ascending=[
                    False,
                    False,
                    True,
                    True,
                ],

                kind="mergesort",
            )
            .reset_index(
                drop=True
            )
        )


        candidate_leaderboard.insert(
            0,
            "rank",
            np.arange(
                1,
                len(candidate_leaderboard) + 1,
            ),
        )


        candidate_leaderboard_path = (
            fold_directory
            / "candidate_leaderboard.csv"
        )


        save_dataframe_atomic(
            candidate_leaderboard,
            candidate_leaderboard_path,
            index=False,
        )


        # ====================================================
        # WINNING CONFIGURATION
        # ====================================================

        winning_row = (
            candidate_leaderboard.iloc[0]
        )


        selected_configuration_id = str(
            winning_row[
                "configuration_id"
            ]
        )

        selected_batch_size = int(
            winning_row[
                "batch_size"
            ]
        )

        selected_head_lr = float(
            winning_row[
                "head_learning_rate"
            ]
        )

        selected_fine_tune_lr = float(
            winning_row[
                "fine_tune_learning_rate"
            ]
        )

        selected_stage = str(
            winning_row[
                "selected_stage"
            ]
        )

        selected_stage_1_epochs = int(
            winning_row[
                "stage_1_best_epoch"
            ]
        )

        selected_stage_2_epochs = int(
            winning_row[
                "stage_2_best_epoch"
            ]
        )


        print(
            "\nSelected configuration:",
            selected_configuration_id,
        )

        print(
            "Batch size:",
            selected_batch_size,
        )

        print(
            "Head learning rate:",
            selected_head_lr,
        )

        print(
            "Fine-tuning learning rate:",
            selected_fine_tune_lr,
        )

        print(
            "Selected stage:",
            selected_stage,
        )

        print(
            "Internal macro F1:",
            f"{winning_row['internal_macro_f1']:.6f}",
        )


        # ====================================================
        # SAVE WINNING CONFIGURATION
        # ====================================================

        selected_configuration = {
            "model":
                "EfficientNetB0",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "configuration_id":
                selected_configuration_id,

            "configuration_number":
                int(
                    winning_row[
                        "configuration_number"
                    ]
                ),

            "batch_size":
                selected_batch_size,

            "head_learning_rate":
                selected_head_lr,

            "fine_tune_learning_rate":
                selected_fine_tune_lr,

            "selected_stage":
                selected_stage,

            "stage_1_refit_epochs":
                selected_stage_1_epochs,

            "stage_2_refit_epochs":
                (
                    selected_stage_2_epochs
                    if selected_stage
                    == "fine_tuning"
                    else 0
                ),

            "internal_macro_f1":
                float(
                    winning_row[
                        "internal_macro_f1"
                    ]
                ),

            "internal_balanced_accuracy":
                float(
                    winning_row[
                        "internal_balanced_accuracy"
                    ]
                ),

            "internal_log_loss":
                float(
                    winning_row[
                        "internal_log_loss"
                    ]
                ),

            "outer_validation_used_for_selection":
                False,
        }


        selected_configuration_path = (
            fold_directory
            / "selected_configuration.json"
        )


        save_json_atomic(
            selected_configuration,
            selected_configuration_path,
        )


        # ====================================================
        # FINAL REFIT ON ALL 4,480 OUTER-TRAINING IMAGES
        # ====================================================

        print(
            "\nRefitting selected configuration "
            "on all 4,480 outer-training images..."
        )


        refit_seed = (
            get_efficientnet_refit_seed(
                repeat_number,
                fold_number,
            )
        )


        tf.keras.backend.clear_session()

        random.seed(
            refit_seed
        )

        np.random.seed(
            refit_seed
        )

        tf.keras.utils.set_random_seed(
            refit_seed
        )

        gc.collect()


        final_training_dataset = (
            make_efficientnetb0_dataset(
                outer_training_dataframe,
                training=True,
                seed=refit_seed,
                batch_size=(
                    selected_batch_size
                ),
            )
        )


        outer_validation_dataset = (
            make_efficientnetb0_dataset(
                outer_validation_dataframe,
                training=False,
                seed=refit_seed,
                batch_size=(
                    selected_batch_size
                ),
            )
        )


        final_model, final_base_model = (
            build_repeated_efficientnetb0_model()
        )


        final_refit_start = (
            time.perf_counter()
        )


        # ====================================================
        # FINAL STAGE 1
        #
        # No internal or outer validation is supplied here.
        # The epoch count was already selected during the
        # internal model-selection procedure.
        # ====================================================

        compile_repeated_efficientnetb0(
            final_model,
            selected_head_lr,
        )


        final_stage_1_start = (
            time.perf_counter()
        )


        final_stage_1_history = (
            final_model.fit(
                final_training_dataset,

                epochs=(
                    selected_stage_1_epochs
                ),

                callbacks=[
                    tf.keras.callbacks
                    .TerminateOnNaN()
                ],

                verbose=0,
            )
        )


        final_stage_1_seconds = (
            time.perf_counter()
            - final_stage_1_start
        )


        final_stage_2_history = None

        final_stage_2_seconds = 0.0


        # ====================================================
        # FINAL STAGE 2 — ONLY IF FINE-TUNING WAS SELECTED
        # ====================================================

        if (
            selected_stage
            == "fine_tuning"
        ):

            enable_repeated_efficientnetb0_fine_tuning(
                final_base_model
            )


            compile_repeated_efficientnetb0(
                final_model,
                selected_fine_tune_lr,
            )


            final_stage_2_start = (
                time.perf_counter()
            )


            final_stage_2_history = (
                final_model.fit(
                    final_training_dataset,

                    epochs=(
                        selected_stage_2_epochs
                    ),

                    callbacks=[
                        tf.keras.callbacks
                        .TerminateOnNaN()
                    ],

                    verbose=0,
                )
            )


            final_stage_2_seconds = (
                time.perf_counter()
                - final_stage_2_start
            )


        final_refit_seconds = (
            time.perf_counter()
            - final_refit_start
        )


        # ====================================================
        # SAVE FINAL REFIT HISTORY
        # ====================================================

        final_stage_1_history_df = pd.DataFrame(
            final_stage_1_history.history
        )


        final_stage_1_history_df.insert(
            0,
            "stage_epoch",
            np.arange(
                1,
                len(
                    final_stage_1_history_df
                ) + 1,
            ),
        )


        final_stage_1_history_df.insert(
            1,
            "stage",
            "frozen_head",
        )


        final_history_frames = [
            final_stage_1_history_df
        ]


        if (
            final_stage_2_history
            is not None
        ):

            final_stage_2_history_df = pd.DataFrame(
                final_stage_2_history.history
            )


            final_stage_2_history_df.insert(
                0,
                "stage_epoch",
                np.arange(
                    1,
                    len(
                        final_stage_2_history_df
                    ) + 1,
                ),
            )


            final_stage_2_history_df.insert(
                1,
                "stage",
                "fine_tuning",
            )


            final_history_frames.append(
                final_stage_2_history_df
            )


        final_refit_history = pd.concat(
            final_history_frames,
            ignore_index=True,
        )


        final_history_path = (
            fold_directory
            / "final_refit_history.csv"
        )


        save_dataframe_atomic(
            final_refit_history,
            final_history_path,
            index=False,
        )


        # ====================================================
        # SAVE FINAL FOLD MODEL WEIGHTS
        # ====================================================

        final_weights_path = (
            fold_directory
            / "final_selected_model.weights.h5"
        )


        temporary_weights_path = (
            fold_directory
            / "final_selected_model.tmp.weights.h5"
        )


        if temporary_weights_path.exists():

            temporary_weights_path.unlink()


        final_model.save_weights(
            temporary_weights_path
        )


        temporary_weights_path.replace(
            final_weights_path
        )


        # ====================================================
        # OUTER VALIDATION — USED ONCE
        # ====================================================

        print(
            "Evaluating untouched outer fold..."
        )


        inference_start = (
            time.perf_counter()
        )


        outer_probabilities = (
            final_model.predict(
                outer_validation_dataset,
                verbose=0,
            )
        )


        inference_seconds = (
            time.perf_counter()
            - inference_start
        )


        outer_probabilities = np.asarray(
            outer_probabilities,
            dtype=np.float32,
        )


        expected_shape = (
            1120,
            NUMBER_OF_CLASSES,
        )


        if (
            outer_probabilities.shape
            != expected_shape
        ):

            raise RuntimeError(
                "Unexpected outer-validation "
                "probability shape."
            )


        if not np.isfinite(
            outer_probabilities
        ).all():

            raise RuntimeError(
                "Non-finite outer-validation "
                "probabilities detected."
            )


        if not np.allclose(
            outer_probabilities.sum(
                axis=1
            ),
            1.0,
            atol=1e-5,
        ):

            raise RuntimeError(
                "Outer probability rows "
                "do not sum to one."
            )


        outer_true_labels = (
            outer_validation_dataframe[
                "class_index"
            ]
            .to_numpy(
                dtype=np.int32
            )
        )


        outer_predicted_labels = (
            np.argmax(
                outer_probabilities,
                axis=1,
            )
            .astype(np.int32)
        )


        outer_metrics = (
            calculate_repeated_efficientnetb0_metrics(
                outer_true_labels,
                outer_predicted_labels,
                outer_probabilities,
            )
        )


        # ====================================================
        # OUTER PREDICTIONS
        # ====================================================

        predictions_dataframe = (
            outer_validation_dataframe[
                [
                    "relative_path",
                    "class",
                    "class_index",
                    "repeat",
                    "fold",
                ]
            ]
            .copy()
        )


        predictions_dataframe = (
            predictions_dataframe.rename(
                columns={
                    "class":
                        "true_class",

                    "class_index":
                        "true_class_index",
                }
            )
        )


        predictions_dataframe[
            "predicted_class_index"
        ] = (
            outer_predicted_labels
        )


        predictions_dataframe[
            "predicted_class"
        ] = [
            INDEX_TO_CLASS[
                int(index)
            ]
            for index
            in outer_predicted_labels
        ]


        predictions_dataframe[
            "correct"
        ] = (
            predictions_dataframe[
                "true_class_index"
            ]
            ==
            predictions_dataframe[
                "predicted_class_index"
            ]
        )


        for (
            class_index,
            class_name,
        ) in enumerate(
            CLASS_NAMES
        ):

            predictions_dataframe[
                f"probability_{class_name}"
            ] = (
                outer_probabilities[
                    :,
                    class_index,
                ]
            )


        predictions_path = (
            fold_directory
            / "outer_validation_predictions.csv"
        )


        save_dataframe_atomic(
            predictions_dataframe,
            predictions_path,
            index=False,
        )


        # ====================================================
        # CONFUSION MATRIX
        # ====================================================

        fold_confusion_matrix = (
            confusion_matrix(
                outer_true_labels,
                outer_predicted_labels,

                labels=list(
                    range(
                        NUMBER_OF_CLASSES
                    )
                ),
            )
        )


        confusion_matrix_dataframe = pd.DataFrame(
            fold_confusion_matrix,
            index=CLASS_NAMES,
            columns=CLASS_NAMES,
        )


        confusion_matrix_path = (
            fold_directory
            / "confusion_matrix.csv"
        )


        save_dataframe_atomic(
            confusion_matrix_dataframe,
            confusion_matrix_path,
            index=True,
        )


        # ====================================================
        # CLASSIFICATION REPORT
        # ====================================================

        fold_report = (
            classification_report(
                outer_true_labels,
                outer_predicted_labels,

                labels=list(
                    range(
                        NUMBER_OF_CLASSES
                    )
                ),

                target_names=(
                    CLASS_NAMES
                ),

                output_dict=True,

                zero_division=0,
            )
        )


        classification_report_dataframe = (
            pd.DataFrame(
                fold_report
            )
            .transpose()
        )


        classification_report_path = (
            fold_directory
            / "classification_report.csv"
        )


        save_dataframe_atomic(
            classification_report_dataframe,
            classification_report_path,
            index=True,
        )


        # ====================================================
        # FINAL OUTER-FOLD METRICS
        # ====================================================

        candidate_search_seconds = float(
            candidate_leaderboard[
                "total_candidate_seconds"
            ].sum()
        )


        fold_metrics = {
            "model":
                "EfficientNetB0",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "outer_split_seed":
                int(
                    partition_manifest[
                        "split_seed"
                    ].iloc[0]
                ),

            "inner_split_seed":
                int(
                    get_efficientnet_inner_split_seed(
                        repeat_number,
                        fold_number,
                    )
                ),

            "candidate_model_seed":
                int(
                    get_efficientnet_model_seed(
                        repeat_number,
                        fold_number,
                    )
                ),

            "final_refit_seed":
                int(
                    refit_seed
                ),

            "selected_configuration_id":
                selected_configuration_id,

            "selected_batch_size":
                selected_batch_size,

            "selected_head_learning_rate":
                selected_head_lr,

            "selected_fine_tune_learning_rate":
                selected_fine_tune_lr,

            "selected_stage":
                selected_stage,

            "selected_stage_1_epochs":
                selected_stage_1_epochs,

            "selected_stage_2_epochs":
                (
                    selected_stage_2_epochs
                    if selected_stage
                    == "fine_tuning"
                    else 0
                ),

            "outer_train_images":
                4480,

            "internal_model_training_images":
                4032,

            "internal_validation_images":
                448,

            "outer_validation_images":
                1120,

            "number_of_candidates":
                12,

            "accuracy":
                float(
                    outer_metrics[
                        "accuracy"
                    ]
                ),

            "balanced_accuracy":
                float(
                    outer_metrics[
                        "balanced_accuracy"
                    ]
                ),

            "macro_precision":
                float(
                    outer_metrics[
                        "macro_precision"
                    ]
                ),

            "macro_recall":
                float(
                    outer_metrics[
                        "macro_recall"
                    ]
                ),

            "macro_f1":
                float(
                    outer_metrics[
                        "macro_f1"
                    ]
                ),

            "log_loss":
                float(
                    outer_metrics[
                        "log_loss"
                    ]
                ),

            "candidate_search_seconds":
                candidate_search_seconds,

            "final_stage_1_seconds":
                float(
                    final_stage_1_seconds
                ),

            "final_stage_2_seconds":
                float(
                    final_stage_2_seconds
                ),

            "final_refit_seconds":
                float(
                    final_refit_seconds
                ),

            "outer_inference_seconds":
                float(
                    inference_seconds
                ),

            "outer_validation_used_for_selection":
                False,

            "testing_images_used":
                False,
        }


        metrics_path = (
            fold_directory
            / "outer_fold_metrics.json"
        )


        save_json_atomic(
            fold_metrics,
            metrics_path,
        )


        # ====================================================
        # VERIFY FOLD OUTPUTS
        # ====================================================

        required_fold_outputs = [
            metrics_path,
            selected_configuration_path,
            predictions_path,
            confusion_matrix_path,
            classification_report_path,
            partition_manifest_path,
            candidate_leaderboard_path,
            final_history_path,
            final_weights_path,
        ]


        if not all(
            path.exists()
            for path
            in required_fold_outputs
        ):

            raise RuntimeError(
                "One or more required "
                "fold outputs are missing."
            )


        saved_predictions = pd.read_csv(
            predictions_path
        )


        if (
            len(
                saved_predictions
            )
            != 1120
        ):

            raise RuntimeError(
                "Saved prediction count "
                "is incorrect."
            )


        # ====================================================
        # FOLD COMPLETE MARKER — WRITTEN LAST
        # ====================================================

        fold_completion = {
            "status":
                "completed",

            "model":
                "EfficientNetB0",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "selected_configuration_id":
                selected_configuration_id,

            "macro_f1":
                float(
                    outer_metrics[
                        "macro_f1"
                    ]
                ),

            "balanced_accuracy":
                float(
                    outer_metrics[
                        "balanced_accuracy"
                    ]
                ),

            "outer_validation_used_once":
                True,

            "outer_validation_used_for_selection":
                False,

            "testing_images_used":
                False,

            "completed_at":
                datetime.now().isoformat(),
        }


        completion_path = (
            fold_directory
            / "fold_complete.json"
        )


        save_json_atomic(
            fold_completion,
            completion_path,
        )


        if not validate_completed_efficientnetb0_fold(
            repeat_number,
            fold_number,
        ):

            raise RuntimeError(
                "Final fold persistence "
                "validation failed."
            )


        print(
            "\nFold completed successfully."
        )

        print(
            "Outer accuracy:",
            f"{outer_metrics['accuracy']:.6f}",
        )

        print(
            "Outer balanced accuracy:",
            f"{outer_metrics['balanced_accuracy']:.6f}",
        )

        print(
            "Outer macro F1:",
            f"{outer_metrics['macro_f1']:.6f}",
        )

        print(
            "Saved:",
            fold_directory,
        )


        # ====================================================
        # CLEAN GPU MEMORY
        # ====================================================

        del final_model
        del final_base_model
        del final_training_dataset
        del outer_validation_dataset
        del outer_probabilities
        del final_stage_1_history

        if (
            final_stage_2_history
            is not None
        ):

            del final_stage_2_history


        tf.keras.backend.clear_session()

        gc.collect()


    # ========================================================
    # 6. AGGREGATE FIVE FOLDS INTO ONE REPEAT RESULT
    # ========================================================

    print(
        "\nAggregating Repeat",
        repeat_number,
    )


    repeat_fold_records = []


    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    ):

        if not validate_completed_efficientnetb0_fold(
            repeat_number,
            fold_number,
        ):

            raise RuntimeError(
                f"Repeat {repeat_number} "
                f"cannot be aggregated because "
                f"Fold {fold_number} is incomplete."
            )


        fold_metrics_path = (
            repeat_directory
            / f"fold_{fold_number:02d}"
            / "outer_fold_metrics.json"
        )


        with fold_metrics_path.open(
            "r",
            encoding="utf-8",
        ) as file:

            repeat_fold_records.append(
                json.load(
                    file
                )
            )


    repeat_fold_metrics = (
        pd.DataFrame(
            repeat_fold_records
        )
        .sort_values(
            "fold"
        )
        .reset_index(
            drop=True
        )
    )


    repeat_fold_metrics_path = (
        repeat_directory
        / "repeat_fold_metrics.csv"
    )


    save_dataframe_atomic(
        repeat_fold_metrics,
        repeat_fold_metrics_path,
        index=False,
    )


    # --------------------------------------------------------
    # ONE statistical observation for this separation/repeat
    # --------------------------------------------------------

    repeat_summary = {
        "status":
            "completed",

        "model":
            "EfficientNetB0",

        "repeat":
            int(
                repeat_number
            ),

        "completed_outer_folds":
            5,

        "mean_accuracy":
            float(
                repeat_fold_metrics[
                    "accuracy"
                ].mean()
            ),

        "sd_accuracy":
            float(
                repeat_fold_metrics[
                    "accuracy"
                ].std(
                    ddof=1
                )
            ),

        "mean_balanced_accuracy":
            float(
                repeat_fold_metrics[
                    "balanced_accuracy"
                ].mean()
            ),

        "sd_balanced_accuracy":
            float(
                repeat_fold_metrics[
                    "balanced_accuracy"
                ].std(
                    ddof=1
                )
            ),

        "mean_macro_precision":
            float(
                repeat_fold_metrics[
                    "macro_precision"
                ].mean()
            ),

        "sd_macro_precision":
            float(
                repeat_fold_metrics[
                    "macro_precision"
                ].std(
                    ddof=1
                )
            ),

        "mean_macro_recall":
            float(
                repeat_fold_metrics[
                    "macro_recall"
                ].mean()
            ),

        "sd_macro_recall":
            float(
                repeat_fold_metrics[
                    "macro_recall"
                ].std(
                    ddof=1
                )
            ),

        "mean_macro_f1":
            float(
                repeat_fold_metrics[
                    "macro_f1"
                ].mean()
            ),

        "sd_macro_f1":
            float(
                repeat_fold_metrics[
                    "macro_f1"
                ].std(
                    ddof=1
                )
            ),

        "mean_log_loss":
            float(
                repeat_fold_metrics[
                    "log_loss"
                ].mean()
            ),

        "sd_log_loss":
            float(
                repeat_fold_metrics[
                    "log_loss"
                ].std(
                    ddof=1
                )
            ),

        "statistical_unit":
            (
                "mean of five outer folds "
                "within this repeat"
            ),

        "wilcoxon_value":
            float(
                repeat_fold_metrics[
                    "macro_f1"
                ].mean()
            ),

        "testing_images_used":
            False,

        "completed_at":
            datetime.now().isoformat(),
    }


    repeat_summary_path = (
        repeat_directory
        / "repeat_summary.json"
    )


    save_json_atomic(
        repeat_summary,
        repeat_summary_path,
    )


    # Update master 10-repeat table immediately.
    update_efficientnetb0_repeat_level_summary()


    print(
        "\n"
        + "-" * 80
    )

    print(
        f"REPEAT {repeat_number:02d} COMPLETED"
    )

    print(
        "-" * 80
    )

    print(
        "Mean five-fold accuracy:",
        f"{repeat_summary['mean_accuracy']:.6f}",
    )

    print(
        "Mean five-fold balanced accuracy:",
        f"{repeat_summary['mean_balanced_accuracy']:.6f}",
    )

    print(
        "Mean five-fold macro F1:",
        f"{repeat_summary['mean_macro_f1']:.6f}",
    )

    print(
        "This is ONE Wilcoxon observation."
    )


# ============================================================
# 7. FINAL 10-REPEAT VALIDATION
# ============================================================

update_efficientnetb0_repeat_level_summary()


final_repeat_summary = pd.read_csv(
    REPEAT_LEVEL_SUMMARY_PATH
)


if len(
    final_repeat_summary
) != 10:

    raise RuntimeError(
        "Expected exactly 10 completed "
        "repeat-level results, but found "
        f"{len(final_repeat_summary)}."
    )


if (
    final_repeat_summary[
        "repeat"
    ].nunique()
    != 10
):

    raise RuntimeError(
        "Repeat-level summary does not "
        "contain 10 unique repeats."
    )


print(
    "\n"
    + "#" * 80
)

print(
    "EFFICIENTNETB0 REPEATED NESTED-CV COMPLETED"
)

print(
    "#" * 80
)

print(
    "Outer evaluations:",
    50,
)

print(
    "Completed repeats:",
    10,
)

print(
    "Wilcoxon observations:",
    len(
        final_repeat_summary
    ),
)

print(
    "\nRepeat-level statistical results:"
)

display(
    final_repeat_summary[
        [
            "repeat",
            "mean_accuracy",
            "mean_balanced_accuracy",
            "mean_macro_f1",
            "sd_macro_f1",
            "wilcoxon_value",
        ]
    ]
)

print(
    "\nSaved to:"
)

print(
    REPEAT_LEVEL_SUMMARY_PATH
)

print(
    "\nTesting images were never used."
)

print(
    "\nStep 13 full repeated EfficientNetB0 experiment PASSED."
)


################################################################################
EFFICIENTNETB0 — REPEAT / SEPARATION 01 OF 10
################################################################################

REPEAT 01 / FOLD 01
Valid completed fold found — skipping.

REPEAT 01 / FOLD 02
Valid completed fold found — skipping.

REPEAT 01 / FOLD 03
Valid completed fold found — skipping.

REPEAT 01 / FOLD 04
Valid completed fold found — skipping.

REPEAT 01 / FOLD 05
Valid completed fold found — skipping.

Aggregating Repeat 1

--------------------------------------------------------------------------------
REPEAT 01 COMPLETED
--------------------------------------------------------------------------------
Mean five-fold accuracy: 0.950893
Mean five-fold balanced accuracy: 0.950893
Mean five-fold macro F1: 0.950996
This is ONE Wilcoxon observation.

################################################################################
EFFICIENTNETB0 — REPEAT / SEPARATION 02 OF 10
#############

,repeat,mean_accuracy,mean_balanced_accuracy,mean_macro_f1,sd_macro_f1,wilcoxon_value
0,1,0.950893,0.950893,0.950996,0.010097,0.950996
1,2,0.955357,0.955357,0.955405,0.012073,0.955405
2,3,0.954286,0.954286,0.954341,0.009514,0.954341
3,4,0.950536,0.950536,0.950651,0.004558,0.950651
4,5,0.958571,0.958571,0.958682,0.002485,0.958682
5,6,0.953214,0.953214,0.953377,0.004670,0.953377
6,7,0.950893,0.950893,0.950888,0.006706,0.950888
7,8,0.953214,0.953214,0.953340,0.010187,0.953340
8,9,0.951607,0.951607,0.951606,0.006015,0.951606
9,10,0.955179,0.955179,0.955386,0.005254,0.955386



Saved to:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0/repeat_level_summary.csv

Testing images were never used.

Step 13 full repeated EfficientNetB0 experiment PASSED.


In [ ]:
# Creating and testing reusable EfficientNetB0 image loader

import numpy as np
import tensorflow as tf

# Image dimensions
IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

# Creating a dictionary that maps each class name to a numerical index
CLASS_TO_INDEX = {class_name: index for index, class_name in enumerate(CLASS_NAMES)}

# Creating the riverse dictionary mapping each numerical index to its class
INDEX_TO_CLASS = {index: class_name for class_name, index in CLASS_TO_INDEX.items()}

# Creating a new column class_index in the assignments dataframe
assignments["class_index"] = assignments["class"].map(CLASS_TO_INDEX)

# Checking for errors
if assignments["class_index"].isna().any():
    unknown_classes = assignments.loc[assignments["class_index"].isna(), "class"].unique()
    raise ValueError(
        f"Unknown classes: {unknown_classes}")


# Converting the values in the class_index column to the np.int32 type
assignments["class_index"] = assignments["class_index"].astype(np.int32)


def load_and_preprocess_efficientnetb0_image(image_path, class_index):
    """
    Loads and preprocesses a PNG image for EfficientNetB0.

    Returns:

    Image: float 32 tensor with shape (224, 224, 3) and values in the 0-255 range.

    Class_index: Integer class label.
    """

    # Reading an image file using the image_path
    image_bytes = tf.io.read_file(image_path)

    # Decoding the raw PNG bytes into actual pixel values
    grayscale_image = tf.io.decode_png(image_bytes, channels=1)


    # Verifying the shape of the grayscale_image
    grayscale_image = tf.ensure_shape(grayscale_image, (IMAGE_HEIGHT, IMAGE_WIDTH, 1))


    # Creating a pseudo-rgb image
    pseudo_rgb_image = tf.image.grayscale_to_rgb(grayscale_image)


    # Verifying the shape of the pseudo_rgb_image
    pseudo_rgb_image = tf.ensure_shape(pseudo_rgb_image, (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))


    # Convert the image pixel values to float32
    efficientnet_image = tf.cast(pseudo_rgb_image, tf.float32)


    # Ensuring the shape of the efficientnet_image has the right shape
    efficientnet_image = tf.ensure_shape(efficientnet_image, (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))


    # Converting the class_index into a 32-bit integer
    class_index = tf.cast(class_index, tf.int32)


    # Returning the preprocessed image and class index
    return efficientnet_image, class_index


# Testing the reusable loader
test_record = assignments.iloc[0]  # the first image from assignments


# Testing the image and the class index
test_image, test_class_index = (
    load_and_preprocess_efficientnetb0_image(
        str(test_record["image_path"]),
        test_record["class_index"]))


# Printing the results
print("--- Class mapping ---")

for class_index in sorted(INDEX_TO_CLASS):
    print(class_index, "->", INDEX_TO_CLASS[class_index])

print("\n--- Loader test ---")
print("Image shape:", test_image.shape)
print("Image data type:", test_image.dtype)
print("Value range:", float(tf.reduce_min(test_image)), "to", float(tf.reduce_max(test_image)))
print("Numeric label:", int(test_class_index))
print("Decoded class:", INDEX_TO_CLASS[int(test_class_index)])
print("Expected class:", test_record["class"])
print("All values finite:", bool(tf.reduce_all(tf.math.is_finite(test_image))))


# Safety checks
if test_image.shape != (224, 224, 3):
    raise ValueError(f"Unexpected image shape: {test_image.shape}")

if int(test_class_index) not in INDEX_TO_CLASS:
    raise ValueError(f"Unexpected class index: {int(test_class_index)}")

if INDEX_TO_CLASS[int(test_class_index)] != test_record["class"]:
    raise ValueError("The numeric label does not match the class.")

if not bool(tf.reduce_all(tf.math.is_finite(test_image))):
    raise ValueError("The image contains NaN or infinite values.")

if float(tf.reduce_min(test_image)) < 0 or float(tf.reduce_max(test_image)) > 255:
    raise ValueError("The image values are not in the 0-255 range.")

print("\nReusable EfficientNetB0 image loader passed.")


In [ ]:
# ---------------------------------------------------------
# EfficientNetB0 TensorFlow datasets for one fold
# ---------------------------------------------------------

import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split

# Candidate batch sizes that will be explored later
BATCH_SIZE_CANDIDATES = [8, 16, 32]

# Use 16 only for testing the dataset pipeline in this cell
TEST_BATCH_SIZE = 16

# Allow TensorFlow to preprocess 2 dataset items at the same time
DATA_PIPELINE_PARALLEL_CALLS = 2

# Prepare 1 upcoming batch in advance while the current batch is being used
PREFETCH_BATCHES = 1

# Set the base seed used to make random operations reproducible
RANDOM_SEED = 42

# Reserve 10% of the outer-training data for internal validation
INTERNAL_VALIDATION_FRACTION = 0.10


def create_efficientnetb0_dataset(dataframe, training, batch_size, shuffle_seed):
    """
    Create a TensorFlow dataset from image paths and labels.
    Training datasets are shuffled.
    Validation datasets retain their fixed order.
    """

    # Getting the path for each image from the dataframe the enters the function as a parameter
    image_paths = dataframe["image_path"].astype(str).to_numpy()

    # Extracting the class index for all the images
    class_indices = dataframe["class_index"].astype(np.int32).to_numpy()

    # Creating a new TensorFlow dataset where each item contains the image path and the class index
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, class_indices))

    # Shuffle the training dataset using a reproducible random seed
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(dataframe),
            seed=shuffle_seed,
            reshuffle_each_iteration=True)

    # Loading and preprocessing each image in the dataset
    dataset = dataset.map(
        load_and_preprocess_efficientnetb0_image,
        num_parallel_calls=DATA_PIPELINE_PARALLEL_CALLS,
        deterministic=True)

    # Grouping the dataset into batches
    dataset = dataset.batch(batch_size, drop_remainder=False)

    # Preparing upcoming batches in advance to reduce waiting during training
    dataset = dataset.prefetch(PREFETCH_BATCHES)

    return dataset


# ---------------------------------------------------------
# Create Fold 1 outer-training and outer-validation tables
# ---------------------------------------------------------

VALIDATION_FOLD = 1

# Select all folds except the validation fold to create the outer-training DataFrame
fold_1_outer_training_df = (
    assignments[assignments["fold"] != VALIDATION_FOLD]
    .copy()
    .reset_index(drop=True))

# Select the validation fold to create the outer-validation DataFrame
fold_1_outer_validation_df = (
    assignments[assignments["fold"] == VALIDATION_FOLD]
    .copy()
    .reset_index(drop=True))


print("--- Fold 1 outer split ---")
print("Outer-training images:", len(fold_1_outer_training_df))
print("Outer-validation images:", len(fold_1_outer_validation_df))


# Checking for errors
if len(fold_1_outer_training_df) != 4480:
    raise ValueError("Fold 1 should contain 4,480 outer-training images.")


if len(fold_1_outer_validation_df) != 1120:
    raise ValueError("Fold 1 should contain 1,120 outer-validation images.")


# ---------------------------------------------------------
# Create deterministic stratified internal validation split
# ---------------------------------------------------------

# Creating a reproducible random seed specific to Fold 1
FOLD_1_RANDOM_SEED = RANDOM_SEED + VALIDATION_FOLD

# We create a training subset and an internal validation subset for fold 1
fold_1_model_training_df, fold_1_internal_validation_df = train_test_split(
    fold_1_outer_training_df,
    test_size=INTERNAL_VALIDATION_FRACTION,
    random_state=FOLD_1_RANDOM_SEED,
    stratify=fold_1_outer_training_df["class_index"])


# Create independent DataFrames and reset their row numbers starting from 0
fold_1_model_training_df = fold_1_model_training_df.copy().reset_index(drop=True)
fold_1_internal_validation_df = fold_1_internal_validation_df.copy().reset_index(drop=True)


print("\n--- Fold 1 internal split ---")
print("Model-training images:", len(fold_1_model_training_df))
print("Internal-validation images:", len(fold_1_internal_validation_df))
print("Outer-validation images:", len(fold_1_outer_validation_df))


# Testing for errors
if len(fold_1_model_training_df) != 4032:
    raise ValueError("Fold 1 should contain 4,032 model-training images.")


if len(fold_1_internal_validation_df) != 448:
    raise ValueError("Fold 1 should contain 448 internal-validation images.")


# ---------------------------------------------------------
# Check class distributions
# ---------------------------------------------------------

print("\n--- Model-training class distribution ---")
print(fold_1_model_training_df["class"].value_counts().sort_index())
print("\n--- Internal-validation class distribution ---")
print(fold_1_internal_validation_df["class"].value_counts().sort_index())
print("\n--- Outer-validation class distribution ---")
print(fold_1_outer_validation_df["class"].value_counts().sort_index())


# ---------------------------------------------------------
# Safety checks for class balance
# ---------------------------------------------------------

# Count the number of model-training images in each class and sort by class name
model_training_counts = fold_1_model_training_df["class"].value_counts().sort_index()

# Count the number of internal-validation images in each class and sort by class name
internal_validation_counts = fold_1_internal_validation_df["class"].value_counts().sort_index()

# Count the number of outer-validation images in each class and sort by class name
outer_validation_counts = fold_1_outer_validation_df["class"].value_counts().sort_index()

# Testing for errors
if not (model_training_counts == 1008).all():
    raise ValueError("Each class should contain 1,008 model-training images.")


if not (internal_validation_counts == 112).all():
    raise ValueError("Each class should contain 112 internal-validation images.")


if not (outer_validation_counts == 280).all():
    raise ValueError("Each class should contain 280 outer-validation images.")


# ---------------------------------------------------------
# Build TensorFlow datasets using one test batch size
# ---------------------------------------------------------

# Create the Fold 1 model-training TensorFlow dataset
fold_1_model_training_dataset = create_efficientnetb0_dataset(
    fold_1_model_training_df,
    training=True,
    batch_size=TEST_BATCH_SIZE,
    shuffle_seed=FOLD_1_RANDOM_SEED)


# Create the Fold 1 internal-validation TensorFlow dataset
fold_1_internal_validation_dataset = create_efficientnetb0_dataset(
    fold_1_internal_validation_df,
    training=False,
    batch_size=TEST_BATCH_SIZE,
    shuffle_seed=FOLD_1_RANDOM_SEED)


# Create the Fold 1 outer-validation TensorFlow dataset
fold_1_outer_validation_dataset = create_efficientnetb0_dataset(
    fold_1_outer_validation_df,
    training=False,
    batch_size=TEST_BATCH_SIZE,
    shuffle_seed=FOLD_1_RANDOM_SEED)


# ---------------------------------------------------------
# Inspect one batch from each dataset
# ---------------------------------------------------------

# Get the first batch of images and labels from the model-training dataset
training_images, training_labels = next(iter(fold_1_model_training_dataset))


# Get the first batch of images and labels from the internal-validation dataset
internal_validation_images, internal_validation_labels = next(iter(fold_1_internal_validation_dataset))


# Get the first batch of images and labels from the outer-validation dataset
outer_validation_images, outer_validation_labels = next(iter(fold_1_outer_validation_dataset))

print("\n--- Model-training batch ---")
print("Image shape:", training_images.shape)
print("Image dtype:", training_images.dtype)
print("Label shape:", training_labels.shape)
print("Label dtype:", training_labels.dtype)
print("Value range:", float(tf.reduce_min(training_images)), "to", float(tf.reduce_max(training_images)))
print("All values finite:", bool(tf.reduce_all(tf.math.is_finite(training_images))))


print("\n--- Internal-validation batch ---")
print("Image shape:", internal_validation_images.shape)
print("Image dtype:", internal_validation_images.dtype)
print("Label shape:", internal_validation_labels.shape)
print("Label dtype:", internal_validation_labels.dtype)
print("Value range:", float(tf.reduce_min(internal_validation_images)), "to", float(tf.reduce_max(internal_validation_images)))
print("All values finite:", bool(tf.reduce_all(tf.math.is_finite(internal_validation_images))))


print("\n--- Outer-validation batch ---")
print("Image shape:", outer_validation_images.shape)
print("Image dtype:", outer_validation_images.dtype)
print("Label shape:", outer_validation_labels.shape)
print("Label dtype:", outer_validation_labels.dtype)
print("Value range:", float(tf.reduce_min(outer_validation_images)), "to", float(tf.reduce_max(outer_validation_images)))
print("All values finite:", bool(tf.reduce_all(tf.math.is_finite(outer_validation_images))))


# ---------------------------------------------------------
# Verify batch shapes
# ---------------------------------------------------------

expected_batch_shape = (TEST_BATCH_SIZE, IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS)

# Testing for errors
if training_images.shape != expected_batch_shape:
    raise ValueError(f"Unexpected model-training batch shape: {training_images.shape}")


if internal_validation_images.shape != expected_batch_shape:
    raise ValueError(f"Unexpected internal-validation batch shape: {internal_validation_images.shape}")


if outer_validation_images.shape != expected_batch_shape:
    raise ValueError(f"Unexpected outer-validation batch shape: {outer_validation_images.shape}")


print("\nFold 1 EfficientNetB0 dataset preparation passed.")

In [ ]:
# ============================================================
# EfficientNetB0 five-fold cross-validation and final training
# ============================================================
#
# Each outer fold is evaluated once and is not used for early stopping.
#
# Completed folds are skipped automatically when this cell is rerun.
#
# Primary controlled experiment:
# No image augmentation is used.
#
# The held-out Testing partition is not used in this cell.

import gc
import json
import os
import random
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, log_loss, matthews_corrcoef,
    precision_recall_fscore_support)

# StratifiedShuffleSplit will create random class-balanced data splits
from sklearn.model_selection import StratifiedShuffleSplit


# ------------------------------------------------------------
# 1. Reproducibility and project paths
# ------------------------------------------------------------

RANDOM_SEED = 42

# Set the hash seed for newly started Python processes to improve reproducibility
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

random.seed(RANDOM_SEED)

# Set NumPy's random seed for reproducible results
np.random.seed(RANDOM_SEED)

tf.keras.utils.set_random_seed(RANDOM_SEED)

# Making TensorFlow operations reproducible
try:
    tf.config.experimental.enable_op_determinism()
    determinism_enabled = True
except Exception as error:
    determinism_enabled = False
    determinism_error = str(error)


PROJECT_ROOT = Path("/content/brain-tumour-mri-classification")

DATA_DIR = PROJECT_ROOT / "processed_data_cropped"

FOLDS_FILE = PROJECT_ROOT / "splits" / "five_fold_cross_validation.csv"

# Where the results will be saved
DRIVE_ROOT = Path("/content/drive/MyDrive/brain_tumour_colab")

ARCHITECTURE_RESULTS_DIR = (DRIVE_ROOT / "results" / "efficientnetb0_transfer_learning")

# Testing for errors
if not PROJECT_ROOT.exists():
    raise FileNotFoundError("The restored project folder was not found: "f"{PROJECT_ROOT}")

if not DATA_DIR.exists():
    raise FileNotFoundError("The processed dataset folder was not found: "f"{DATA_DIR}")

if not FOLDS_FILE.exists():
    raise FileNotFoundError("The fixed fold file was not found: "f"{FOLDS_FILE}")


# ------------------------------------------------------------
# 2. Experiment configuration
# ------------------------------------------------------------

# Name of the experiment with the five-fold cross-validation
EXPERIMENT_NAME = ("efficientnetb0_hyperparameter_search_no_augmentation_v2")

# Name of the final model trained on all the training images
FINAL_EXPERIMENT_NAME = ("efficientnetb0_selected_final_no_augmentation_v2")

OUTER_FOLDS = [1, 2, 3, 4, 5]

# Reserve 10% of each outer-training partition for internal validation
INNER_VALIDATION_FRACTION = 0.10

# Train the new classification head for up to 15 epochs
HEAD_EPOCHS = 15

# Fine-tune the selected EfficientNetB0 layers for up to 20 epochs
FINE_TUNE_EPOCHS = 20

# Batch sizes that will be explored
BATCH_SIZE_CANDIDATES = [8, 16, 32]

# Classifier-head learning rates that will be explored
HEAD_LEARNING_RATE_CANDIDATES = [1e-3, 3e-4]

# Fine-tuning learning rates that will be explored
FINE_TUNE_LEARNING_RATE_CANDIDATES = [1e-5, 3e-6]


# Create every combination of batch size, classifier-head learning rate,
# and fine-tuning learning rate that will be explored
HYPERPARAMETER_CONFIGURATIONS = []

# Going through each batch size
for batch_size in BATCH_SIZE_CANDIDATES:

    # Going through each classifier-head learning rate
    for head_learning_rate in HEAD_LEARNING_RATE_CANDIDATES:

        # Going through each fine-tuning learning rate
        for fine_tune_learning_rate in FINE_TUNE_LEARNING_RATE_CANDIDATES:

            # Create a readable name for this hyperparameter configuration
            configuration_id = (
                "efficientnetb0"
                f"_bs{batch_size}"
                f"_headlr{head_learning_rate:.0e}"
                f"_ftlr{fine_tune_learning_rate:.0e}"
            )

            # Store all settings belonging to this configuration
            HYPERPARAMETER_CONFIGURATIONS.append(
                {
                    "configuration_id": configuration_id,
                    "batch_size": batch_size,
                    "head_learning_rate": head_learning_rate,
                    "fine_tune_learning_rate": fine_tune_learning_rate,
                }
            )


# Check that the expected 12 configurations were created
if len(HYPERPARAMETER_CONFIGURATIONS) != 12:
    raise RuntimeError("Expected 12 hyperparameter configurations, "f"but found {len(HYPERPARAMETER_CONFIGURATIONS)}.")

DROPOUT_RATE = 0.30

# Stop training after 3 epochs without validation-loss improvement
EARLY_STOPPING_PATIENCE = 3

# Fine-tune EfficientNetB0's final block7 stage.
FINE_TUNE_FROM_LAYER = ("block7a_expand_conv")

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

# Create mappings between class names and their numerical class indices
CLASS_TO_INDEX = {class_name: class_index for class_index, class_name in enumerate(CLASS_NAMES)}
INDEX_TO_CLASS = {class_index: class_name for class_name, class_index in CLASS_TO_INDEX.items()}


# Use standard float32 for the primary experiment.
tf.keras.mixed_precision.set_global_policy("float32")


# Create the folder path for the five-fold hyperparameter-search results
CV_RESULTS_DIR = (ARCHITECTURE_RESULTS_DIR / EXPERIMENT_NAME)

# Create the folder path for the selected final model-training results
FINAL_RESULTS_DIR = (ARCHITECTURE_RESULTS_DIR / FINAL_EXPERIMENT_NAME)

# Create the cross-validation results folder and any missing parent folders
CV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Create the final-training results folder and any missing parent folders
FINAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# Record the hyperparameter-search experiment settings for reproducibility
SEARCH_CONFIGURATION = {
    "experiment_name": EXPERIMENT_NAME,
    "architecture": "EfficientNetB0",
    "pretrained_weights": "ImageNet",
    "input_shape": [IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS],
    "input_value_range": "0-255 float32",
    "class_names": CLASS_NAMES,
    "batch_size_candidates": (BATCH_SIZE_CANDIDATES),
    "outer_folds": OUTER_FOLDS,
    "inner_validation_fraction": (INNER_VALIDATION_FRACTION),
    "maximum_head_epochs": (HEAD_EPOCHS),
    "maximum_fine_tune_epochs": (FINE_TUNE_EPOCHS),
    "head_learning_rate_candidates": (HEAD_LEARNING_RATE_CANDIDATES),
    "fine_tune_learning_rate_candidates": (FINE_TUNE_LEARNING_RATE_CANDIDATES),
    "number_of_configurations": (
        len(BATCH_SIZE_CANDIDATES)
        * len(HEAD_LEARNING_RATE_CANDIDATES)
        * len(FINE_TUNE_LEARNING_RATE_CANDIDATES)
    ),
    "dropout_rate": DROPOUT_RATE,
    "fine_tune_from_layer": (FINE_TUNE_FROM_LAYER),
    "batch_normalisation_frozen": True,
    "data_augmentation": False,
    "random_seed": RANDOM_SEED,
    "tensorflow_version": tf.__version__,
    "keras_version": tf.keras.__version__,
}

# Save the hyperparameter-search experiment settings as a readable JSON file
with open(CV_RESULTS_DIR / "search_configuration.json", "w", encoding="utf-8") as file:
    json.dump(SEARCH_CONFIGURATION, file, indent=2)



# ------------------------------------------------------------
# 3. Load the fixed fold assignments
# ------------------------------------------------------------

# Load the fixed fold assignments from the CSV file
assignments = pd.read_csv(FOLDS_FILE)


# Define the columns that must exist in the fold-assignment file
required_columns = {"relative_path", "class", "fold",}


# Find any required columns that are missing from the fold file
missing_columns = required_columns - set(assignments.columns)


# Stop the program if required columns are missing
if missing_columns:raise ValueError(f"The fold file is missing these columns: {sorted(missing_columns)}")


# Convert each class name into its numerical class index
assignments["class_index"] = assignments["class"].map(CLASS_TO_INDEX)


# Create the full file path for each image from its relative path
assignments["image_path"] = (
    assignments["relative_path"].map(
        lambda relative_path: str(DATA_DIR / relative_path)))


# From this point onwards we test for errors. The comments inside the raise part describe what we are testing for
if len(assignments) != 5600:
    raise ValueError("Expected 5,600 fold assignments, "f"but found {len(assignments)}.")


if assignments["relative_path"].duplicated().any():
    raise ValueError("Duplicate image paths exist in the fold file.")


if assignments["class_index"].isna().any():
    raise ValueError("An unexpected class exists in the fold file.")


# Create a list of image paths that do not exist on disk
missing_image_paths = [path for path in assignments["image_path"] if not Path(path).exists()]


if missing_image_paths:
    raise FileNotFoundError(f"{len(missing_image_paths)} assigned images are missing.")


# Convert the class-index column to 32-bit integers
assignments["class_index"] = assignments["class_index"].astype("int32")


if sorted(assignments["fold"].unique()) != OUTER_FOLDS:
    raise ValueError("The fold file does not contain the expected folds 1 to 5.")

if not (assignments.groupby("fold").size()== 1120).all():
    raise ValueError("Every outer fold must contain 1,120 images.")


if not (assignments.groupby(["fold", "class"]).size() == 280).all():
    raise ValueError("Every outer fold must contain 280 images from each class.")


# ------------------------------------------------------------
# 4. EfficientNetB0 image-loading function
# ------------------------------------------------------------

def load_and_preprocess_efficientnetb0_image(image_path, class_index):
    """
    Load one stored grayscale PNG and prepare it for EfficientNetB0.

    EfficientNetB0 receives float32 pseudo-RGB pixels in the original 0-255 range because the Keras model
    contains its own input-rescaling layer.
    """

    # Reading the image as a raw file using the image path
    image_bytes = tf.io.read_file(image_path)


    # Decode the PNG bytes into a one-channel grayscale image tensor
    grayscale_image = tf.io.decode_png(image_bytes,channels=1)


    # Confirm that the greyscale image has the expected dimensions
    grayscale_image = tf.ensure_shape(grayscale_image, (IMAGE_HEIGHT, IMAGE_WIDTH, 1))


    # Changing the image to a three channel pseudo-rgb image
    pseudo_rgb_image = tf.image.grayscale_to_rgb(grayscale_image)


    # Convert the image pixel values from integers to float32
    efficientnet_image = tf.cast(pseudo_rgb_image,tf.float32)


    # Checking the image shape again
    efficientnet_image = tf.ensure_shape(efficientnet_image, (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))


    # Convert the class label to a 32-bit integer
    class_index = tf.cast(class_index, tf.int32)

    return efficientnet_image, class_index


# ------------------------------------------------------------
# 5. TensorFlow dataset function
# ------------------------------------------------------------

def make_dataset(dataframe, training, seed, batch_size):
    """
    Convert a partition DataFrame into a deterministic TensorFlow dataset.
    """

    # Convert the image-path column into a NumPy array of strings
    image_paths = dataframe["image_path"].astype(str).to_numpy()


    # Convert the class-index column into a NumPy array of int32 labels
    class_indices = dataframe["class_index"].astype("int32").to_numpy()


    # Pair each image path with its class label to create a TensorFlow dataset
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, class_indices))


    # Shuffle training data in a reproducible new order each epoch
    if training:
        dataset = dataset.shuffle(len(dataframe), seed=seed, reshuffle_each_iteration=True)

    # Load and preprocess dataset images in parallel while preserving their order
    dataset = dataset.map(
        load_and_preprocess_efficientnetb0_image,
        num_parallel_calls=(tf.data.AUTOTUNE),
        deterministic=True)

    # Group images and labels into batches, keeping the final smaller batch
    dataset = dataset.batch(batch_size, drop_remainder=False)


    # Prepare upcoming batches in advance to reduce training delays
    dataset = dataset.prefetch(tf.data.AUTOTUNE)


    # Create a dataset-options object for controlling pipeline behaviour and make it reproducible
    options = tf.data.Options()
    options.experimental_deterministic = (True)

    return dataset.with_options(options)

# ------------------------------------------------------------
# 6. Create training, stopping and outer partitions
# ------------------------------------------------------------

def make_partitions(outer_fold):
    """
    Create three non-overlapping partitions:

    1. Model-training data
    2. Inner early-stopping data
    3. Untouched outer-validation data
    """

    # Select the current fold as the untouched outer-validation partition
    outer_validation = (assignments[assignments["fold"] == outer_fold]
        .copy()
        .reset_index(drop=True))


    # Select all remaining folds as the outer-training partition
    outer_training = (assignments[assignments["fold"] != outer_fold]
        .copy()
        .reset_index(drop=True))


    # Configure one reproducible, class-balanced internal training split
    splitter = (
        StratifiedShuffleSplit(n_splits=1, test_size=(INNER_VALIDATION_FRACTION),
            random_state=(RANDOM_SEED + outer_fold)))


    # Generate row positions for the class-balanced training and early-stopping partitions
    training_positions, early_stopping_positions = next(
        splitter.split(outer_training, outer_training["class_index"]))


    # Select the rows assigned to the model-training partition
    model_training = (outer_training.iloc[training_positions]
        .copy()
        .reset_index(drop=True))


    # Select the rows assigned to the early-stopping partition
    early_stopping = (outer_training
        .iloc[early_stopping_positions]
        .copy()
        .reset_index(drop=True))


    # Label every row as belonging to the model-training partition
    model_training["partition"] = ("model_training")


    # Label every row as belonging to the early-stopping partition
    early_stopping["partition"] = ("early_stopping")


    # Label every row as belonging to the outer-validation partition
    outer_validation["partition"] = ("outer_validation")


    # Combine all three partitions into one complete dataframe named manifest
    manifest = pd.concat([model_training, early_stopping, outer_validation], ignore_index=True)


    # Expected size of the manifest
    expected_sizes = (4032,448,1120)


    # Actual sizes of the manifest
    actual_sizes = (len(model_training), len(early_stopping), len(outer_validation))


    # Checking for errors
    if actual_sizes != expected_sizes:
        raise RuntimeError("Unexpected partition sizes. "f"Found {actual_sizes}; "f"expected {expected_sizes}.")


    if manifest["relative_path"].duplicated().any():
        raise RuntimeError("The training, early-stopping and outer partitions overlap.")


    return (model_training, early_stopping, outer_validation, manifest)


# ------------------------------------------------------------
# 7. EfficientNetB0 model functions
# ------------------------------------------------------------

def build_model():
    """
    Build a frozen ImageNet EfficientNetB0 and a four-class softmax classifier.
    """

    # Define the expected shape of each input image
    inputs = tf.keras.Input(
        shape=(IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS), name="input_image")

    # The base model with the ImageNet weights used for features extraction
    base_model = (
        tf.keras.applications.EfficientNetB0(
            include_top=False,
            weights="imagenet",
            input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS),
            pooling="avg"))


    base_model.trainable = False

    # Features extracted
    features = base_model(inputs, training=False)


    # Apply dropout to reduce overfitting in the classifier
    features = tf.keras.layers.Dropout(rate=DROPOUT_RATE, name="classifier_dropout")(features)


    # Creating the final classification layer and converting the extracted features into four class probabilities
    outputs = tf.keras.layers.Dense(
        units=NUMBER_OF_CLASSES,
        activation="softmax",
        dtype="float32",
        name="class_probabilities")(features)


    model = tf.keras.Model(inputs = inputs, outputs = outputs, name=("efficientnetb0_transfer_learning"))

    return (model, base_model)

# Configure the model's optimizer, loss function, and evaluation metric
def compile_model(model, learning_rate):

    model.compile(
        optimizer=(         # The optimiser chosen
            tf.keras.optimizers.Adam(learning_rate=(learning_rate))),

        loss=(              # The loss chosen
            tf.keras.losses
            .SparseCategoricalCrossentropy()
        ),
        metrics=[           # Track the percentage of images classified correctly
            tf.keras.metrics
            .SparseCategoricalAccuracy(name="accuracy")])


def enable_fine_tuning(base_model):
    """
    Unfreeze EfficientNetB0's final block7 stage.
    Batch-normalisation layers remain frozen.
    """

    base_model.trainable = True

    # Track whether the fine-tuning starting layer has been reached
    start_found = False

    # going over the layers of the model
    for layer in base_model.layers:

        if (layer.name == FINE_TUNE_FROM_LAYER):
            start_found = True

        # Unfreeze layers from the chosen starting point, except batch-normalisation layers
        layer.trainable = start_found and not isinstance(layer, tf.keras.layers.BatchNormalization)


    if not start_found:
        raise ValueError("Fine-tuning layer was not found: "f"{FINE_TUNE_FROM_LAYER}")


    # Collect the names of all EfficientNetB0 layers enabled for fine-tuning
    trainable_layers = [layer.name for layer in base_model.layers if layer.trainable]


    if not trainable_layers:
        raise RuntimeError("No EfficientNetB0 layers were enabled for fine-tuning.")

    return trainable_layers

# Create the callbacks used to control and monitor model training
def create_callbacks(checkpoint_path, minimum_learning_rate):

    return [
        # Save the model weights whenever validation loss reaches a new minimum
        tf.keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
        # Stop training when validation loss stops improving and restore the best weights
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=(EARLY_STOPPING_PATIENCE),
            restore_best_weights=True,
            verbose=1,
        ),
        # Reduce the learning rate when validation loss stops improving
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.2,
            patience=2,
            min_lr=minimum_learning_rate,
            verbose=1,
        ),
        # Stop training immediately if the loss becomes NaN
        tf.keras.callbacks.TerminateOnNaN(),
    ]


# ------------------------------------------------------------
# 8. Classification metric function
# ------------------------------------------------------------
# Calculate classification metrics from the true labels, predictions, and probabilities

def calculate_metrics(y_true, y_predicted, probabilities):

    # Calculate equally weighted precision, recall, and F1 across all classes
    macro_scores = precision_recall_fscore_support(y_true, y_predicted, average="macro", zero_division=0)

    # Calculate precision, recall, and F1 weighted by the number of images in each class
    weighted_scores = precision_recall_fscore_support(y_true, y_predicted, average="weighted", zero_division=0)

    # returning the metrics calculated
    return {
        "accuracy": float(accuracy_score(y_true, y_predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_predicted)),
        "macro_precision": float(macro_scores[0]),
        "macro_recall": float(macro_scores[1]),
        "macro_f1": float(macro_scores[2]),
        "weighted_precision": float(weighted_scores[0]),
        "weighted_recall": float(weighted_scores[1]),
        "weighted_f1": float(weighted_scores[2]),
        "matthews_correlation_coefficient": (float(matthews_corrcoef(y_true, y_predicted))),
        "log_loss": float(log_loss(y_true, probabilities, labels=list(range(NUMBER_OF_CLASSES))))
    }


# ------------------------------------------------------------
# 9. Verify that the GPU is still available
# ------------------------------------------------------------

gpu_devices = (tf.config.list_physical_devices("GPU"))

if not gpu_devices:
    raise RuntimeError("No TensorFlow GPU is available. Reconnect to a GPU runtime before training.")


print("=" * 70)
print("EfficientNetB0 five-fold cross-validation")
print("=" * 70)
print("Python version:", sys.version)
print("TensorFlow version:", tf.__version__)
print("Keras version:", tf.keras.__version__)
print("GPU devices:", gpu_devices)
print("Deterministic operations enabled:", determinism_enabled)

if not determinism_enabled:
    print("Determinism error:",determinism_error)

print("Results directory:", CV_RESULTS_DIR)
print("Data augmentation:", False)
print("Outer folds:", OUTER_FOLDS)


# ------------------------------------------------------------
# 10. Sequential hyperparameter search and five-fold training
# ------------------------------------------------------------
# Run the complete five-fold training and validation process for every hyperparameter configuration


# Going over each hyperparameter configuration
for configuration in HYPERPARAMETER_CONFIGURATIONS:

    # Get the settings belonging to the current configuration
    configuration_id = configuration["configuration_id"]
    current_batch_size = configuration["batch_size"]
    current_head_learning_rate = configuration["head_learning_rate"]
    current_fine_tune_learning_rate = configuration["fine_tune_learning_rate"]

    # Create a separate results directory for the current hyperparameter configuration
    configuration_directory = CV_RESULTS_DIR / configuration_id

    # Create the configuration results directory and any missing parent folders
    configuration_directory.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 70)
    print("HYPERPARAMETER CONFIGURATION:", configuration_id)
    print("Batch size:", current_batch_size)
    print("Classifier-head learning rate:", current_head_learning_rate)
    print("Fine-tuning learning rate:", current_fine_tune_learning_rate)
    print("=" * 70)

    # Going over each fold
    for outer_fold in OUTER_FOLDS:

        # Build the results-directory path for the current outer fold
        fold_directory = configuration_directory / f"fold_{outer_fold}"


        # Create the fold results directory, including missing parent directories
        fold_directory.mkdir(parents=True, exist_ok=True)


        # Build the file path for the current fold's saved metrics
        metrics_file = fold_directory / "fold_metrics.json"


        # Build the file path for the current fold's outer-validation predictions
        predictions_file = fold_directory / "outer_validation_predictions.csv"


        # Build the file path used to mark that the current fold completed successfully
        completion_file = fold_directory / "fold_complete.json"


        # List the fold outputs that must exist before a completed fold can be skipped
        required_fold_output_files = [
            fold_directory / "experiment_configuration.json",
            fold_directory / "partition_manifest.csv",
            fold_directory / "training_history.csv",
            metrics_file,
            predictions_file,
            fold_directory / "confusion_matrix.csv",
            fold_directory / "classification_report.csv",
            fold_directory / "timing.json",
            fold_directory / "selected_best.weights.h5",
        ]

        # Skip a fold only when its completion marker and required outputs all exist.
        if (completion_file.exists() and all(path.exists()for path in required_fold_output_files)):
            print(f"\nFold {outer_fold} already completed — skipping.")
            continue


        # Remove incomplete fold results so that partial checkpoints are never reused
        if completion_file.exists() or any(path.exists() for path in required_fold_output_files):
            print(f"\nIncomplete results found for Fold {outer_fold} — restarting this fold.")
            shutil.rmtree(fold_directory)
            fold_directory.mkdir(parents=True, exist_ok=True)


        # Verify that a TensorFlow GPU is still available before starting the current configuration-fold run
        current_gpu_devices = tf.config.list_physical_devices("GPU")

        if not current_gpu_devices:
            raise RuntimeError("No TensorFlow GPU is available. ",
                               f"Training cannot continue for configuration {configuration_id}, ",
                               f"Fold {outer_fold}.")



        print("\n" + "-" * 70)
        print(f"OUTER FOLD {outer_fold}")
        print("-" * 70)


        # Record the time when processing for the current fold begins
        fold_start_time = time.perf_counter()

        # Create a reproducible random seed that is different for each outer fold
        fold_seed = RANDOM_SEED + outer_fold

        # Store the complete experiment settings used for the current configuration and fold
        fold_experiment_configuration = {
            "experiment_name": EXPERIMENT_NAME,
            "architecture": "EfficientNetB0",
            "configuration_id": configuration_id,
            "outer_fold": outer_fold,
            "fold_seed": fold_seed,
            "batch_size": current_batch_size,
            "head_learning_rate": current_head_learning_rate,
            "fine_tune_learning_rate": current_fine_tune_learning_rate,
            "maximum_head_epochs": HEAD_EPOCHS,
            "maximum_fine_tune_epochs": FINE_TUNE_EPOCHS,
            "inner_validation_fraction": INNER_VALIDATION_FRACTION,
            "dropout_rate": DROPOUT_RATE,
            "fine_tune_from_layer": FINE_TUNE_FROM_LAYER,
            "batch_normalisation_frozen": True,
            "data_augmentation": False,
            "input_shape": [IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS],
            "input_value_range": "0-255 float32",
            "class_names": CLASS_NAMES,
            "random_seed": RANDOM_SEED,
            "testing_partition_used": False,
        }


        # Save the current configuration and fold settings as a readable JSON file
        with open(fold_directory / "experiment_configuration.json", "w", encoding="utf-8") as file:
            json.dump(fold_experiment_configuration, file, indent=2)

        # Clear any previously created Keras models and free their stored state
        tf.keras.backend.clear_session()


        # Set TensorFlow, Keras, NumPy, and Python random seeds for this fold
        tf.keras.utils.set_random_seed(fold_seed)


        # Ask Python to remove unused objects from memory
        gc.collect()


        # Create the training, early-stopping, outer-validation, and combined partition DataFrames
        model_training_dataframe, early_stopping_dataframe, outer_validation_dataframe, partition_manifest = make_partitions(outer_fold)

        # Save the current fold's image partition assignments as a CSV file
        partition_manifest[["relative_path", "class", "class_index", "fold", "partition"]].to_csv(
            fold_directory / "partition_manifest.csv", index=False)


        print("Model-training images:", len(model_training_dataframe))
        print("Early-stopping images:", len(early_stopping_dataframe))
        print("Outer-validation images:", len(outer_validation_dataframe))

        training_dataset = make_dataset(
            model_training_dataframe,
            training=True,
            seed=fold_seed,
            batch_size=current_batch_size)


        early_stopping_dataset = make_dataset(
            early_stopping_dataframe,
            training=False,
            seed=fold_seed,
            batch_size=current_batch_size)


        outer_validation_dataset = make_dataset(
            outer_validation_dataframe,
            training=False,
            seed=fold_seed,
            batch_size=current_batch_size)

        # This calls and stores the complete model and pretrained base model separately
        model, base_model = build_model()

        # Save the model architecture summary to a text file in the current fold directory
        with open(fold_directory / "model_summary.txt", "w", encoding="utf-8") as summary_file:
            model.summary(print_fn=lambda line: summary_file.write(line + "\n"))


        # --------------------------------------------------------
        # Stage 1: frozen EfficientNetB0 base
        # --------------------------------------------------------

        # Create the file path for saving the best weights while training only the classifier head
        stage_1_checkpoint = str(fold_directory / "stage_1_best.weights.h5")


        compile_model(model, current_head_learning_rate)
        print("\nStage 1 of 2 — training classifier head")

        # Record the time when classifier-head training begins
        stage_1_start_time = time.perf_counter()


        # Training only the classifier head using model.fit() and store its training and validation history
        stage_1_history = model.fit(
            training_dataset,
            validation_data=(early_stopping_dataset),
            epochs=HEAD_EPOCHS,
            callbacks=create_callbacks(stage_1_checkpoint, minimum_learning_rate=1e-6),
            verbose=1)


        # Calculate how many seconds classifier-head training took
        stage_1_training_seconds = time.perf_counter() - stage_1_start_time


        # Find and store the lowest validation loss reached while training only the classifier head
        stage_1_best_validation_loss = float(min(stage_1_history.history["val_loss"]))


        # Find and store the epoch number with the lowest validation loss during classifier-head training
        stage_1_best_epoch = int(np.argmin(stage_1_history.history["val_loss"]) + 1)



        # --------------------------------------------------------
        # Stage 2: fine-tune EfficientNetB0's final stage
        # --------------------------------------------------------


        # Explicitly restore the best Stage 1 checkpoint before beginning fine-tuning
        model.load_weights(stage_1_checkpoint)


        # Unfreeze selected EfficientNetB0 layers for fine-tuning and store which base layers are now trainable
        trainable_base_layers = enable_fine_tuning(base_model)


        # Create the file path for saving the best weights while fine-tuning selected EfficientNetB0 layers
        stage_2_checkpoint = str(fold_directory / "stage_2_best.weights.h5")


        compile_model(model, current_fine_tune_learning_rate)

        print("\nStage 2 of 2 — fine-tuning block7")
        print("Trainable base layers:", len(trainable_base_layers))


        # Starting time for when the fine-tuning begins
        stage_2_start_time = time.perf_counter()


        # Fine-tune the selected EfficientNetB0 layers and store the training and validation history
        stage_2_history = model.fit(
            training_dataset,
            validation_data=(early_stopping_dataset),
            epochs=FINE_TUNE_EPOCHS,
            callbacks=create_callbacks(stage_2_checkpoint, minimum_learning_rate=1e-7),
            verbose=1)

        # Time taken for the stage two training
        stage_2_training_seconds = time.perf_counter() - stage_2_start_time


        # Find and store the lowest validation loss reached during fine-tuning
        stage_2_best_validation_loss = float(min(stage_2_history.history["val_loss"]))


        # Find and store the epoch number with the lowest validation loss during fine-tuning
        stage_2_best_epoch = int(np.argmin(stage_2_history.history["val_loss"]) + 1)


        # --------------------------------------------------------
        # Select the better internal-validation checkpoint
        # --------------------------------------------------------

        # Create the file path for storing whichever checkpoint performed best across both training stages
        selected_checkpoint = fold_directory / "selected_best.weights.h5"


        # Select the checkpoint with the lowest validation loss across both training stages
        if stage_2_best_validation_loss < stage_1_best_validation_loss:
            selected_stage = "fine_tuning"
            selected_source = Path(stage_2_checkpoint)
            selected_validation_loss = stage_2_best_validation_loss
            selected_epoch = stage_2_best_epoch

        else:
            selected_stage = "frozen_head"
            selected_source = Path(stage_1_checkpoint)
            selected_validation_loss = stage_1_best_validation_loss
            selected_epoch = stage_1_best_epoch

            # Restore the frozen EfficientNetB0 base state because Stage 1 was selected
            base_model.trainable = False


        # Copy the best-performing weights file to the selected checkpoint path
        shutil.copy2(selected_source, selected_checkpoint)


        # Load the best-performing weights into the model
        model.load_weights(str(selected_checkpoint))


        print("\nSelected stage:", selected_stage)
        print("Selected internal validation loss:", selected_validation_loss)


        # --------------------------------------------------------
        # Save training histories
        # --------------------------------------------------------

        # Convert the classifier-head training history into a pandas DataFrame
        stage_1_history_dataframe = pd.DataFrame(stage_1_history.history)


        # Add epoch numbers as the first column
        stage_1_history_dataframe.insert(0, "epoch", np.arange(1, len(stage_1_history_dataframe) + 1))


        # Add the training stage as the second column
        stage_1_history_dataframe.insert(1, "stage", "frozen_head")


        # Convert the fine-tuning history into a DataFrame
        stage_2_history_dataframe = pd.DataFrame(stage_2_history.history)


        # Add epoch numbers as the first column
        stage_2_history_dataframe.insert(0, "epoch", np.arange(1, len(stage_2_history_dataframe) + 1))


        # Add the fine-tuning stage as the second column
        stage_2_history_dataframe.insert(1, "stage", "fine_tuning")


        # Concatinating the stage1 and stage2 history
        pd.concat([stage_1_history_dataframe, stage_2_history_dataframe], ignore_index=True,
                  ).to_csv(fold_directory / "training_history.csv", index=False)


        # Save the fine-tuning settings for the current fold
        with open(
            fold_directory / "fine_tuning_configuration.json", "w", encoding="utf-8",
        ) as file:
            json.dump(
                {
                    "fine_tune_from_layer": (FINE_TUNE_FROM_LAYER),
                    "trainable_base_layers": (trainable_base_layers),
                    "batch_normalisation_frozen": (True),
                },
                file,
                indent=2,
            )


        # --------------------------------------------------------
        # Evaluate once on the untouched outer fold
        # --------------------------------------------------------

        print("\nEvaluating untouched "f"outer Fold {outer_fold}...")

        # Record the time when evaluation on the untouched outer fold begins
        inference_start_time = (time.perf_counter())


        # Use the trained model to predict class probabilities for the untouched outer-validation images
        probabilities = model.predict(outer_validation_dataset, verbose=1)


        # Time taken for the evaluation
        inference_seconds = time.perf_counter() - inference_start_time


        # Get the true class labels for the untouched outer-validation images
        true_labels = outer_validation_dataframe["class_index"].to_numpy(dtype="int32")


        # Choose the class with the highest predicted probability for each image
        predicted_labels = np.argmax(probabilities, axis=1).astype("int32") # probabilities contains four probabilities for each image


        # Checking that the number of images is the same in both probabilities and outer_validation_dataframe
        if len(probabilities) != len(outer_validation_dataframe):
            raise RuntimeError("The prediction count does not match the outer-fold image count.")


        # Calculate the evaluation metrics with the function we defined earlier for the untouched outer-validation fold
        fold_metrics = calculate_metrics(true_labels, predicted_labels, probabilities)


        # Adding extra information about the metrics used
        fold_metrics.update({
            "configuration_id": configuration_id,
            "batch_size": current_batch_size,
            "head_learning_rate": current_head_learning_rate,
            "fine_tune_learning_rate": current_fine_tune_learning_rate,
            "outer_fold": outer_fold,
            "selected_stage": selected_stage,
            "selected_best_epoch": selected_epoch,
            "selected_inner_validation_loss": selected_validation_loss,
            "stage_1_best_epoch": stage_1_best_epoch,
            "stage_1_best_validation_loss": stage_1_best_validation_loss,
            "stage_2_best_epoch": stage_2_best_epoch,
            "stage_2_best_validation_loss": stage_2_best_validation_loss,
            "stage_1_training_seconds": stage_1_training_seconds,
            "stage_2_training_seconds": stage_2_training_seconds,
            "total_training_seconds": stage_1_training_seconds + stage_2_training_seconds,
            "inference_seconds": inference_seconds,
            "milliseconds_per_image": inference_seconds / len(outer_validation_dataframe) * 1000,
            "fold_total_seconds": time.perf_counter() - fold_start_time,
        })


        # Store the training and inference timing information for the current fold
        fold_timing = {
            "configuration_id": configuration_id,
            "outer_fold": outer_fold,
            "stage_1_training_seconds": stage_1_training_seconds,
            "stage_2_training_seconds": stage_2_training_seconds,
            "total_training_seconds": stage_1_training_seconds + stage_2_training_seconds,
            "inference_seconds": inference_seconds,
            "milliseconds_per_image": inference_seconds / len(outer_validation_dataframe) * 1000,
            "fold_total_seconds": fold_metrics["fold_total_seconds"],
        }


        # Save the current fold's timing information as a readable JSON file
        with open(fold_directory / "timing.json", "w", encoding="utf-8") as file:
            json.dump(fold_timing, file, indent=2)


        # Copy the main image information for the outer-validation where we will later add predictions
        predictions_dataframe = outer_validation_dataframe[["relative_path", "class", "class_index", "fold"]].copy()


        # Rename the true label columns so they are clearly different from the model's predictions
        predictions_dataframe = predictions_dataframe.rename(columns={"class": "true_class", "class_index": "true_class_index"})


        # Add the hyperparameter configuration used to create these predictions
        predictions_dataframe["configuration_id"] = configuration_id


        # Add the predicted class index for each image
        predictions_dataframe["predicted_class_index"] = predicted_labels


        # Convert each predicted class index into its class name
        predictions_dataframe["predicted_class"] = [CLASS_NAMES[index] for index in predicted_labels]


        # Starting a loop over all the class names
        for class_index, class_name in enumerate(CLASS_NAMES):
            # Add a column for this class and fill it with that class’s predicted probability for every image.
            predictions_dataframe[f"probability_{class_name}"] = probabilities[:, class_index]


        # Saving the predictions dataframe as a CSV file
        predictions_dataframe.to_csv(predictions_file, index=False)


        # Create the confusion matrix by comparing the true and predicted classes
        fold_confusion_matrix = confusion_matrix(true_labels, predicted_labels, labels=list(range(NUMBER_OF_CLASSES)))


        # Create a labeled confusion matrix DataFrame and save it as a CSV file
        pd.DataFrame(fold_confusion_matrix, index=CLASS_NAMES, columns=CLASS_NAMES,
        ).to_csv(fold_directory / "confusion_matrix.csv", index_label="true_class")


        # Compare the true and predicted labels and store a detailed per-class performance report in fold_report
        fold_report = (
            classification_report(
                true_labels,
                predicted_labels,
                labels=list(range(NUMBER_OF_CLASSES)),
                target_names=CLASS_NAMES,
                output_dict=True,
                zero_division=0))


        # Turning the fold_report into a dataframe then saving it as a CSV file
        pd.DataFrame(fold_report).transpose().to_csv(fold_directory / "classification_report.csv",
            index_label=("class_or_average"))


        # Keep only the selected checkpoint by deleting the temporary Stage 1 and Stage 2 checkpoints.
        for temporary_checkpoint in [Path(stage_1_checkpoint), Path(stage_2_checkpoint)]:
            if temporary_checkpoint.exists():
                temporary_checkpoint.unlink()


        # Save the current fold's metrics as a readable JSON file
        with open(metrics_file, "w", encoding="utf-8") as file:
            json.dump(fold_metrics, file, indent=2)


        # This completion marker is written last.
        with open(completion_file, "w", encoding="utf-8",) as file:
            json.dump(
                {
                    "completed": True,
                    "configuration_id": configuration_id,
                    "outer_fold": outer_fold,
                    "completed_at_unix_time": time.time(),
                },
                file,
                indent=2)


        print(f"\nFold {outer_fold} completed.")
        print("Accuracy:", f"{fold_metrics['accuracy']:.4f}")
        print("Macro F1:", f"{fold_metrics['macro_f1']:.4f}")
        print("Selected stage:", selected_stage)
        print("Saved to:", fold_directory)


        # Model objects we no longer needed for this fold and can be deleted from memory
        del model
        del base_model
        del training_dataset
        del early_stopping_dataset
        del outer_validation_dataset
        del stage_1_history
        del stage_2_history
        del probabilities


        # Clear Keras’s old model state before moving to the next fold.
        tf.keras.backend.clear_session()


        # Clean up unused Python memory before starting the next fold.
        gc.collect()


# ------------------------------------------------------------
# 11. Aggregate all five outer-fold results for every configuration
# ------------------------------------------------------------

# Create an empty list to store the summary results from every hyperparameter configuration
configuration_metric_records = []


# Going through each hyperparameter configuration
for configuration in HYPERPARAMETER_CONFIGURATIONS:

    # Get the settings belonging to the current configuration
    configuration_id = configuration["configuration_id"]
    current_batch_size = configuration["batch_size"]
    current_head_learning_rate = configuration["head_learning_rate"]
    current_fine_tune_learning_rate = configuration["fine_tune_learning_rate"]

    # Get the results directory belonging to the current configuration
    configuration_directory = CV_RESULTS_DIR / configuration_id


    print("\n" + "=" * 70)
    print("AGGREGATING CONFIGURATION:", configuration_id)
    print("=" * 70)


    # Create a list of the metrics file paths for all five folds
    metric_files = [
        configuration_directory / f"fold_{fold}" / "fold_metrics.json"
        for fold in OUTER_FOLDS
    ]

    # Create a list of the prediction file paths for all five folds
    prediction_files = [
        configuration_directory / f"fold_{fold}" / "outer_validation_predictions.csv"
        for fold in OUTER_FOLDS
    ]

    # Create a list of any missing metrics or prediction files
    missing_output_files = [
        str(path) for path in (metric_files + prediction_files)
        if not path.exists()
    ]

    # Dealing with errors in case of missing files
    if missing_output_files:
        raise RuntimeError("Aggregation could not run because some fold outputs are missing:\n" + "\n".join(missing_output_files))


    # Create an empty list to store the metrics from all five folds
    fold_metric_records = []

    # Going through each fold's metrics file.
    for metric_file in metric_files:

        # Opens that JSON file so Python can read it
        with open(metric_file, "r", encoding="utf-8") as file:
            fold_metric_records.append(   # Adding the file to the list created earlier
                json.load(file))


    # Creating a dataframe containing the metrics from all five folds
    all_fold_metrics = (
        pd.DataFrame(fold_metric_records).sort_values("outer_fold").reset_index(drop=True))

    # Read and combine the prediction files from all five folds into one DataFrame
    all_outer_fold_predictions = pd.concat([pd.read_csv(path) for path in prediction_files],
        ignore_index=True)


    # Testing for length error
    if len(all_outer_fold_predictions) != 5600:
        raise RuntimeError("Expected 5,600 out-of-fold predictions, but found "f"{len(all_outer_fold_predictions)}.")


    # Testing for duplication
    if all_outer_fold_predictions["relative_path"].duplicated().any():
        raise RuntimeError("Duplicate out-of-fold predictions were found.")


    # Create a list of the probability column names for all four classes
    probability_columns = [f"probability_{class_name}" for class_name in CLASS_NAMES]


    # Get the true class indices for all out-of-fold images as a NumPy array
    out_of_fold_true_labels = all_outer_fold_predictions["true_class_index"].to_numpy(dtype="int32")


    # Get the predicted class indices for all out-of-fold images as a NumPy array
    out_of_fold_predicted_labels = (
        all_outer_fold_predictions["predicted_class_index"].to_numpy(dtype="int32"))


    # The predicted class probabilities for all out-of-fold images as a NumPy array
    out_of_fold_probabilities = all_outer_fold_predictions[probability_columns].to_numpy(dtype="float32")


    # Calculate the overall metrics using the combined predictions from all five folds
    pooled_out_of_fold_metrics = calculate_metrics(
        out_of_fold_true_labels,
        out_of_fold_predicted_labels,
        out_of_fold_probabilities,
    )


    # Metrics calculated
    metric_names = [
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_precision",
        "weighted_recall",
        "weighted_f1",
        "matthews_correlation_coefficient",
        "log_loss",
    ]


    # Create an empty list to store the cross-validation summary for each metric
    summary_rows = []

    # Store the mean, standard deviation, and pooled value for each evaluation metric
    for metric_name in metric_names:

        summary_rows.append(
            {
                "metric": metric_name,
                "fold_mean": float(all_fold_metrics[metric_name].mean()),
                "fold_standard_deviation": float(all_fold_metrics[metric_name].std(ddof=1)),
                "pooled_out_of_fold_value": float(pooled_out_of_fold_metrics[metric_name]),
            }
        )


    # Convert the cross-validation summary rows into a DataFrame
    cross_validation_summary = pd.DataFrame(summary_rows)


    # Save the metrics from all five folds as a CSV file
    all_fold_metrics.to_csv(configuration_directory / "all_fold_metrics.csv", index=False)


    # Save all out-of-fold predictions as a CSV file
    all_outer_fold_predictions.to_csv(configuration_directory / "all_out_of_fold_predictions.csv", index=False)


    # Save the cross-validation summary as a CSV file
    cross_validation_summary.to_csv(configuration_directory / "cross_validation_summary.csv", index=False)


    # creates a 4 x4 table showing: true class vs predicted class for all 5,600 out-of-fold images
    pooled_confusion_matrix = confusion_matrix(
        out_of_fold_true_labels,
        out_of_fold_predicted_labels,
        labels=list(range(NUMBER_OF_CLASSES)))


    # Converting the pooled confusion matrix into a dataframe then saving it as a CSV file
    pd.DataFrame(pooled_confusion_matrix, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
        configuration_directory / "out_of_fold_confusion_matrix.csv", index_label="true_class")


    # Create the classification report using the combined predictions from all five folds
    pooled_report = classification_report(
        out_of_fold_true_labels,
        out_of_fold_predicted_labels,
        labels=list(range(NUMBER_OF_CLASSES)),
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0)


    # Converting the report into a dataframe and saving it as a csv file
    pd.DataFrame(pooled_report).transpose().to_csv(
        configuration_directory / "out_of_fold_classification_report.csv",
        index_label=("class_or_average"))


    # Save the combined out-of-fold metrics as a JSON file
    with open(configuration_directory / "pooled_out_of_fold_metrics.json", "w", encoding="utf-8") as file:
        json.dump(pooled_out_of_fold_metrics, file, indent=2)


    # Store the hyperparameters belonging to the current configuration
    configuration_summary = {
        "configuration_id": configuration_id,
        "batch_size": current_batch_size,
        "head_learning_rate": current_head_learning_rate,
        "fine_tune_learning_rate": current_fine_tune_learning_rate,
    }


    # Store the mean, standard deviation, and pooled value for each metric
    for metric_name in metric_names:

        configuration_summary[f"mean_{metric_name}"] = float(all_fold_metrics[metric_name].mean())

        configuration_summary[f"standard_deviation_{metric_name}"] = float(all_fold_metrics[metric_name].std(ddof=1))

        configuration_summary[f"pooled_{metric_name}"] = float(pooled_out_of_fold_metrics[metric_name])


    # Store the total training time across all five folds
    configuration_summary[
        "total_training_seconds"
    ] = float(all_fold_metrics["total_training_seconds"].sum())


    # Store the average inference time per image across all five folds
    configuration_summary[
        "mean_milliseconds_per_image"
    ] = float(all_fold_metrics["milliseconds_per_image"].mean())


    # Add the completed configuration summary to the full experiment results
    configuration_metric_records.append(configuration_summary)


    print("Configuration completed:", configuration_id)
    print("Mean accuracy:", f"{configuration_summary['mean_accuracy']:.4f}")
    print("Mean balanced accuracy:", f"{configuration_summary['mean_balanced_accuracy']:.4f}")
    print("Mean macro F1:", f"{configuration_summary['mean_macro_f1']:.4f}")
    print("Macro F1 standard deviation:", f"{configuration_summary['standard_deviation_macro_f1']:.4f}")
    print("Mean log loss:", f"{configuration_summary['mean_log_loss']:.4f}")


# Create one DataFrame containing the five-fold results for every hyperparameter configuration
all_configuration_metrics = (
    pd.DataFrame(configuration_metric_records).reset_index(drop=True))


# Save the complete hyperparameter-search results as a CSV file
all_configuration_metrics.to_csv(CV_RESULTS_DIR / "all_configuration_metrics.csv", index=False)


print("\n" + "=" * 70)
print("ALL HYPERPARAMETER CONFIGURATIONS AGGREGATED")
print("=" * 70)
print("Configurations completed:", len(all_configuration_metrics))
print("Results saved to:", CV_RESULTS_DIR)


# ------------------------------------------------------------
# 12. Select the best hyperparameter configuration
# ------------------------------------------------------------

# Rank the hyperparameter configurations using the predefined selection rule
hyperparameter_leaderboard = (
    all_configuration_metrics
    .sort_values(
        by=[
            "mean_macro_f1",
            "mean_balanced_accuracy",
            "standard_deviation_macro_f1",
            "mean_log_loss",
        ],
        ascending=[False, False, True, True])
    .reset_index(drop=True))

# Add a ranking number starting from 1 for the best configuration
hyperparameter_leaderboard.insert(0, "rank", np.arange(1, len(hyperparameter_leaderboard) + 1))


# Select the first row because it is the best configuration after ranking
selected_configuration_row = (hyperparameter_leaderboard.iloc[0])


# Store the selected hyperparameter values
SELECTED_CONFIGURATION_ID = str(selected_configuration_row["configuration_id"])

SELECTED_BATCH_SIZE = int(selected_configuration_row["batch_size"])

SELECTED_HEAD_LEARNING_RATE = float(selected_configuration_row["head_learning_rate"])

SELECTED_FINE_TUNE_LEARNING_RATE = float(selected_configuration_row["fine_tune_learning_rate"])


# Create the results directory belonging to the selected configuration
SELECTED_CONFIGURATION_DIR = (CV_RESULTS_DIR / SELECTED_CONFIGURATION_ID)


# Store the selected configuration and its main selection metrics
selected_configuration = {
    "configuration_id": (SELECTED_CONFIGURATION_ID),
    "batch_size": (SELECTED_BATCH_SIZE),
    "head_learning_rate": (SELECTED_HEAD_LEARNING_RATE),
    "fine_tune_learning_rate": (SELECTED_FINE_TUNE_LEARNING_RATE),
    "mean_macro_f1": float(selected_configuration_row["mean_macro_f1"]),
    "mean_balanced_accuracy": float(selected_configuration_row["mean_balanced_accuracy"]),
    "standard_deviation_macro_f1": float(selected_configuration_row["standard_deviation_macro_f1"]),
    "mean_log_loss": float(selected_configuration_row["mean_log_loss"]),
    "selection_rule": (
        "Highest mean five-fold macro F1; "
        "then highest mean balanced accuracy; "
        "then lower macro-F1 standard deviation; "
        "then lower mean log loss."
    ),
}


# Save the complete ranked hyperparameter leaderboard as a CSV file
hyperparameter_leaderboard.to_csv(
    CV_RESULTS_DIR / "hyperparameter_leaderboard.csv", index=False)


# Save the selected hyperparameter configuration as a readable JSON file
with open(CV_RESULTS_DIR / "selected_configuration.json", "w", encoding="utf-8") as file:
    json.dump(selected_configuration, file, indent=2)


print("\n" + "=" * 70)
print("SELECTED HYPERPARAMETER CONFIGURATION")
print("=" * 70)
print("Configuration:", SELECTED_CONFIGURATION_ID)
print("Batch size:", SELECTED_BATCH_SIZE)
print("Classifier-head learning rate:", SELECTED_HEAD_LEARNING_RATE)
print("Fine-tuning learning rate:", SELECTED_FINE_TUNE_LEARNING_RATE)
print("Mean macro F1:", f"{selected_configuration['mean_macro_f1']:.4f}")
print("Mean balanced accuracy:", f"{selected_configuration['mean_balanced_accuracy']:.4f}")
print("Macro F1 standard deviation:", f"{selected_configuration['standard_deviation_macro_f1']:.4f}")
print("Mean log loss:", f"{selected_configuration['mean_log_loss']:.4f}")



# ------------------------------------------------------------
# 13. Select final training durations from the selected configuration
# ------------------------------------------------------------

# Load the five fold metrics belonging only to the selected hyperparameter configuration
selected_fold_metrics = pd.read_csv(SELECTED_CONFIGURATION_DIR / "all_fold_metrics.csv")


# Check that all five outer folds are available for selecting the final training durations
if len(selected_fold_metrics) != 5:
    raise RuntimeError("Expected metrics from 5 folds for the selected configuration, "f"but found {len(selected_fold_metrics)}.")


# Look at the best classifier-head epoch across the five folds
# and use the median as the Stage 1 training duration for the final model.
FINAL_HEAD_EPOCHS = max(1, int(np.rint(selected_fold_metrics["stage_1_best_epoch"].median())))


# Look at the best fine-tuning epoch across the five folds
# and use the median as the Stage 2 training duration for the final model.
FINAL_FINE_TUNE_EPOCHS = max(1, int(np.rint(selected_fold_metrics["stage_2_best_epoch"].median())))


# Count how many folds selected each training stage as the better checkpoint
stage_selection_counts = (selected_fold_metrics["selected_stage"].value_counts())


# Select the stage chosen by the majority of the five folds
FINAL_SELECTED_STAGE = str(stage_selection_counts.idxmax())


# Store exactly how the final epoch counts and selected training stage were chosen
selected_epoch_summary = {
    "configuration_id": (SELECTED_CONFIGURATION_ID),

    "epoch_selection_rule": (
        "Use the rounded median best epoch across "
        "the five folds of the selected configuration "
        "separately for Stage 1 and Stage 2."),

    "stage_selection_rule": (
        "Use the training stage selected by the majority "
        "of the five folds."),

    "fold_stage_1_best_epochs": (selected_fold_metrics["stage_1_best_epoch"].astype(int).tolist()),

    "fold_stage_2_best_epochs": (selected_fold_metrics["stage_2_best_epoch"].astype(int).tolist()),

    "fold_selected_stages": (selected_fold_metrics["selected_stage"].tolist()),

    "selected_head_epochs": (FINAL_HEAD_EPOCHS),

    "selected_fine_tune_epochs": (FINAL_FINE_TUNE_EPOCHS),

    "selected_final_stage": (FINAL_SELECTED_STAGE),
}


# Save the selected final training epochs and how they were chosen as a JSON file
with open(CV_RESULTS_DIR / "selected_epoch_summary.json", "w", encoding="utf-8") as file:
    json.dump(selected_epoch_summary, file, indent=2)


print("\n" + "=" * 70)
print("SELECTED FINAL TRAINING DURATIONS")
print("=" * 70)
print("Selected configuration:", SELECTED_CONFIGURATION_ID)
print("Stage 1 best epochs across folds:", selected_epoch_summary["fold_stage_1_best_epochs"])
print("Selected final head epochs:", FINAL_HEAD_EPOCHS)
print("Stage 2 best epochs across folds:", selected_epoch_summary["fold_stage_2_best_epochs"])
print("Selected final fine-tuning epochs:", FINAL_FINE_TUNE_EPOCHS)
print("Stages selected across folds:", selected_epoch_summary["fold_selected_stages"])
print("Selected final stage:", FINAL_SELECTED_STAGE)



# Check that all expected hyperparameter configurations were successfully aggregated
if len(all_configuration_metrics) != len(HYPERPARAMETER_CONFIGURATIONS):
    raise RuntimeError("The hyperparameter search cannot be marked complete because ",
        f"{len(all_configuration_metrics)} of ",
        f"{len(HYPERPARAMETER_CONFIGURATIONS)} configurations were aggregated.")


# Create a list containing every expected fold-completion marker
expected_fold_completion_files = [
    CV_RESULTS_DIR
    / configuration["configuration_id"]
    / f"fold_{outer_fold}"
    / "fold_complete.json"
    for configuration in HYPERPARAMETER_CONFIGURATIONS
    for outer_fold in OUTER_FOLDS
]


# Find any folds that do not have a completion marker
missing_fold_completion_files = [
    str(path)
    for path in expected_fold_completion_files
    if not path.exists()
]


# Stop if any of the 60 configuration-fold runs are incomplete
if missing_fold_completion_files:
    raise RuntimeError(
        "The hyperparameter search cannot be marked complete because "
        "some fold-completion markers are missing:\n"
        + "\n".join(missing_fold_completion_files))


# Create the architecture-level search completion marker
SEARCH_COMPLETION_FILE = (CV_RESULTS_DIR / "search_completion.json")


# This completion marker is written after the full search and selection process has completed
with open(SEARCH_COMPLETION_FILE, "w", encoding="utf-8",
) as file:

    json.dump(
        {
            "completed": True,
            "architecture": "EfficientNetB0",
            "number_of_configurations": len(HYPERPARAMETER_CONFIGURATIONS),
            "number_of_outer_folds": len(OUTER_FOLDS),
            "number_of_configuration_fold_runs": len(HYPERPARAMETER_CONFIGURATIONS) * len(OUTER_FOLDS),
            "selected_configuration_id": SELECTED_CONFIGURATION_ID,
            "selected_batch_size": SELECTED_BATCH_SIZE,
            "selected_head_learning_rate": SELECTED_HEAD_LEARNING_RATE,
            "selected_fine_tune_learning_rate": SELECTED_FINE_TUNE_LEARNING_RATE,
            "selected_head_epochs": FINAL_HEAD_EPOCHS,
            "selected_fine_tune_epochs": FINAL_FINE_TUNE_EPOCHS,
            "selected_final_stage": FINAL_SELECTED_STAGE,
            "testing_partition_used": False,
            "completed_at_unix_time": time.time(),
        },
        file,
        indent=2,
    )


print("Hyperparameter search completion marker:", SEARCH_COMPLETION_FILE)




# ------------------------------------------------------------
# 14. Train a fresh final model on all 5,600 Training images
# ------------------------------------------------------------

# Create the file path where the final trained EfficientNetB0 model will be saved
FINAL_MODEL_PATH = (FINAL_RESULTS_DIR / "final_efficientnetb0_cropped.keras")

# Create the file path used to mark that final model training completed successfully
FINAL_COMPLETION_FILE = (FINAL_RESULTS_DIR / "training_complete.json")

# List the final-training outputs that must exist before final training can be skipped
required_final_output_files = [
    FINAL_MODEL_PATH,
    FINAL_RESULTS_DIR / "combined_training_history.csv",
    FINAL_RESULTS_DIR / "final_training_manifest.csv",
    FINAL_RESULTS_DIR / "final_training_configuration.json",
    FINAL_RESULTS_DIR / "final_training_timing.json",
    FINAL_RESULTS_DIR / "trainable_layer_manifest.json",
]


# Skip final training only if the completion marker and all required outputs exist
if FINAL_COMPLETION_FILE.exists() and all(path.exists() for path in required_final_output_files):
    print("\nFinal EfficientNetB0 training already completed — skipping.")
    print("Final model:", FINAL_MODEL_PATH)

else:
    print("\n" + "=" * 70)
    print("FINAL EFFICIENTNETB0 TRAINING ON ALL 5,600 TRAINING IMAGES")
    print("=" * 70)
    print("Selected configuration:", SELECTED_CONFIGURATION_ID)
    print("Batch size:", SELECTED_BATCH_SIZE)
    print("Classifier-head learning rate:", SELECTED_HEAD_LEARNING_RATE)
    print("Fine-tuning learning rate:", SELECTED_FINE_TUNE_LEARNING_RATE)
    print("Final head epochs:", FINAL_HEAD_EPOCHS)
    print("Final fine-tuning epochs:", FINAL_FINE_TUNE_EPOCHS)
    print("Final selected stage:", FINAL_SELECTED_STAGE)

    # Clear any previous Keras model state before building the final model
    tf.keras.backend.clear_session()

    # Set the random seed so final training is reproducible
    tf.keras.utils.set_random_seed(RANDOM_SEED)

    # Clean up any unused Python memory before final training
    gc.collect()


    # Create the shuffled training dataset using all 5,600 images
    final_training_dataset = make_dataset(assignments, training=True, seed=RANDOM_SEED, batch_size=SELECTED_BATCH_SIZE)


    # Build a new final model and its pretrained EfficientNetB0 base model
    final_model, final_base_model = build_model()

    # Open a text file where the final model architecture summary will be saved
    with open(FINAL_RESULTS_DIR / "model_summary.txt", "w", encoding="utf-8") as summary_file:
        final_model.summary(print_fn=lambda line: summary_file.write(line + "\n"))


    # --------------------------------------------------------
    # Final Stage 1: train the classification head
    # --------------------------------------------------------

    # Compile the final model using the selected classifier-head learning rate
    compile_model(final_model, SELECTED_HEAD_LEARNING_RATE)

    print("\nFinal Stage 1 of 2 — training classifier head")
    print("Epochs:", FINAL_HEAD_EPOCHS)

    # Record the time when final classifier-head training begins
    final_stage_1_start_time = time.perf_counter()

    # Train the final model's classifier head on all training images and store the training history
    final_stage_1_history = final_model.fit(
        final_training_dataset,
        epochs=FINAL_HEAD_EPOCHS,
        callbacks=[tf.keras.callbacks.TerminateOnNaN()],
        verbose=1)


    # Calculate how many seconds final classifier-head training took
    final_stage_1_training_seconds = time.perf_counter() - final_stage_1_start_time

    # Save the classifier-head training weights
    final_model.save_weights(FINAL_RESULTS_DIR / "stage_1_head_training.weights.h5")


    # --------------------------------------------------------
    # Final Stage 2: fine-tune the final EfficientNetB0 stage
    # --------------------------------------------------------

    # Create default values in case the selected final model stops after Stage 1
    final_stage_2_history = None
    final_stage_2_training_seconds = 0.0
    final_trainable_base_layers = []


    # Fine-tune the selected EfficientNetB0 layers only if fine-tuning was selected across the five folds
    if FINAL_SELECTED_STAGE == "fine_tuning":

        # Unfreeze the selected EfficientNetB0 layers for final fine-tuning
        final_trainable_base_layers = enable_fine_tuning(final_base_model)

        # Recompile the final model using the selected smaller fine-tuning learning rate
        compile_model(final_model, SELECTED_FINE_TUNE_LEARNING_RATE)

        print("\nFinal Stage 2 of 2 — fine-tuning block7")
        print("Epochs:", FINAL_FINE_TUNE_EPOCHS)
        print("Trainable base layers:", len(final_trainable_base_layers))

        # Record the time when final fine-tuning begins
        final_stage_2_start_time = time.perf_counter()

        # Fine-tune the selected EfficientNetB0 layers on all training images and store the training history
        final_stage_2_history = final_model.fit(
            final_training_dataset,
            epochs=FINAL_FINE_TUNE_EPOCHS,
            callbacks=[tf.keras.callbacks.TerminateOnNaN()],
            verbose=1)


        # Calculate how many seconds final fine-tuning took
        final_stage_2_training_seconds = (time.perf_counter() - final_stage_2_start_time)

        # Save the final fine-tuned model weights
        final_model.save_weights(FINAL_RESULTS_DIR / "stage_2_fine_tuned.weights.h5")

    else:
        print("\nFinal Stage 2 skipped because the frozen-head stage was selected across the five folds.")


    # Save the complete final trained model
    final_model.save(FINAL_MODEL_PATH)


    # --------------------------------------------------------
    # Save final training histories and configuration
    # --------------------------------------------------------

    # Convert the final classifier-head training history into a pandas DataFrame
    final_stage_1_history_dataframe = pd.DataFrame(final_stage_1_history.history)

    # Add epoch numbers as the first column
    final_stage_1_history_dataframe.insert(0, "epoch", np.arange(1, len(final_stage_1_history_dataframe) + 1))


    # Add the training stage as the second column
    final_stage_1_history_dataframe.insert(1, "stage", "frozen_head")


    # Create a list containing the training-history DataFrames that were actually used
    final_training_history_dataframes = [final_stage_1_history_dataframe]


    # Add the Stage 2 history only if fine-tuning was performed
    if final_stage_2_history is not None:

        # Convert the final fine-tuning history into a pandas DataFrame
        final_stage_2_history_dataframe = pd.DataFrame(final_stage_2_history.history)

        # Add epoch numbers as the first column
        final_stage_2_history_dataframe.insert(0, "epoch", np.arange(
                1,
                len(final_stage_2_history_dataframe) + 1))

        # Add the fine-tuning stage as the second column
        final_stage_2_history_dataframe.insert(1, "stage", "fine_tuning")

        # Add the Stage 2 history to the list
        final_training_history_dataframes.append(final_stage_2_history_dataframe)


    # Combine the training histories that were actually used and save them as one CSV file
    pd.concat(
        final_training_history_dataframes,
        ignore_index=True,
    ).to_csv(FINAL_RESULTS_DIR / "combined_training_history.csv", index=False)


    # Save the final training image list with each image's class and fold assignment
    assignments[
        [
            "relative_path",
            "class",
            "class_index",
            "fold",
        ]
    ].to_csv(FINAL_RESULTS_DIR / "final_training_manifest.csv", index=False)


    # Store the final model architecture, training settings, dataset details, and software versions in one configuration dictionary
    final_training_configuration = {
        "experiment_name": (FINAL_EXPERIMENT_NAME),
        "selected_configuration_id": (SELECTED_CONFIGURATION_ID),
        "architecture": "EfficientNetB0",
        "pretrained_weights": "ImageNet",
        "training_images": 5600,
        "input_shape": [IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS],
        "input_value_range": "0-255 float32",
        "class_names": CLASS_NAMES,
        "batch_size": (SELECTED_BATCH_SIZE),
        "head_epochs": (FINAL_HEAD_EPOCHS),
        "fine_tune_epochs": (FINAL_FINE_TUNE_EPOCHS),
        "head_learning_rate": (SELECTED_HEAD_LEARNING_RATE),
        "fine_tune_learning_rate": (SELECTED_FINE_TUNE_LEARNING_RATE),
        "selected_final_stage": (FINAL_SELECTED_STAGE),
        "dropout_rate": DROPOUT_RATE,
        "fine_tune_from_layer": (FINE_TUNE_FROM_LAYER),
        "trainable_base_layers": (final_trainable_base_layers),
        "batch_normalisation_frozen": True,
        "data_augmentation": False,
        "random_seed": RANDOM_SEED,
        "testing_partition_used": False,
        "tensorflow_version": tf.__version__,
        "keras_version": tf.keras.__version__,
    }


    # Save the final training configuration as a readable JSON file
    with open(FINAL_RESULTS_DIR / "final_training_configuration.json", "w", encoding="utf-8") as file:
        json.dump(final_training_configuration, file, indent=2)


    # Store the Stage 1, Stage 2, and total final training times
    final_training_timing = {
        "stage_1_training_seconds": (final_stage_1_training_seconds),
        "stage_2_training_seconds": (final_stage_2_training_seconds),
        "total_training_seconds": (final_stage_1_training_seconds + final_stage_2_training_seconds)
    }


    # Save the final training times as a readable JSON file
    with open(FINAL_RESULTS_DIR / "final_training_timing.json", "w", encoding="utf-8") as file:
        json.dump(final_training_timing, file, indent=2)


    # Save the final EfficientNetB0 trainable-layer manifest
    with open(FINAL_RESULTS_DIR / "trainable_layer_manifest.json", "w", encoding="utf-8") as file:
        json.dump(
            {
                "selected_final_stage": (FINAL_SELECTED_STAGE),
                "fine_tune_from_layer": (FINE_TUNE_FROM_LAYER),
                "trainable_base_layers": (final_trainable_base_layers),
                "batch_normalisation_frozen": True,
            },
            file,
            indent=2)


    # This completion marker is written last.
    with open(FINAL_COMPLETION_FILE, "w", encoding="utf-8") as file:
        json.dump(
            {
                "completed": True,
                "model_path": str(FINAL_MODEL_PATH),
                "selected_configuration_id": (SELECTED_CONFIGURATION_ID),
                "selected_final_stage": (FINAL_SELECTED_STAGE),
                "testing_partition_used": False,
                "completed_at_unix_time": time.time(),
            },
            file,
            indent=2)


print("\n" + "=" * 70)
print("FINAL EFFICIENTNETB0 TRAINING COMPLETED")
print("=" * 70)
print("Final model:", FINAL_MODEL_PATH)
print("Selected configuration:", SELECTED_CONFIGURATION_ID)
print("Selected final stage:", FINAL_SELECTED_STAGE)
print("Cross-validation results:", CV_RESULTS_DIR)
print("Final training results:", FINAL_RESULTS_DIR)
print("Held-out Testing partition used:", False)




In [ ]:
# =========================================================
# VERIFY THE HELD-OUT TESTING PARTITION
# =========================================================

from pathlib import Path
import pandas as pd


# ---------------------------------------------------------
# 1. Define the Testing directory
# ---------------------------------------------------------

# Create the path to the cropped Testing image directory used for final model verification
TESTING_DIRECTORY = (PROJECT_ROOT / "processed_data_cropped" / "Testing")

print("Testing directory:", TESTING_DIRECTORY)
print("Testing directory exists:", TESTING_DIRECTORY.exists())

if not TESTING_DIRECTORY.exists():
    raise FileNotFoundError(f"Testing directory not found: {TESTING_DIRECTORY}")


# ---------------------------------------------------------
# 2. Create the Testing manifest
# ---------------------------------------------------------

# This list will store information about every testing image
testing_records = []

# Going through each class in the Testing dataset to collect every testing image
for class_name in CLASS_NAMES:

    # Creates the path to the current Testing class folder
    class_directory = (TESTING_DIRECTORY / class_name)

    # Checking for errors
    if not class_directory.exists():
        raise FileNotFoundError(f"Testing class directory not found: "f"{class_directory}")

    # Get the numeric class index for the current Testing class
    class_index = CLASS_TO_INDEX[class_name]

    # Go through every image in the current Testing class and store its path and true class information
    for image_path in sorted(class_directory.glob("*.png")):
        testing_records.append(
            {
                "relative_path": (image_path.relative_to(DATA_DIR).as_posix()),
                "class": class_name,
                "class_index": class_index,
                "image_path": str(image_path),
            })


# Convert all stored Testing image records into a pandas DataFrame
testing_manifest = pd.DataFrame(testing_records)


# ---------------------------------------------------------
# 3. Inspect the Testing partition
# ---------------------------------------------------------

print("\n--- Testing partition ---")
print("Total images:", len(testing_manifest))
print("Unique image paths:", testing_manifest["relative_path"].nunique())
print("\nImages per class:")
print(testing_manifest["class"].value_counts().reindex(CLASS_NAMES))

expected_testing_class_counts = {
    "glioma": 400,
    "meningioma": 400,
    "notumor": 398,
    "pituitary": 400,
}

actual_testing_class_counts = (
    testing_manifest["class"]
    .value_counts()
    .reindex(CLASS_NAMES)
    .to_dict()
)

if actual_testing_class_counts != expected_testing_class_counts:
    raise ValueError( "Unexpected Testing class counts. "
        f"Found {actual_testing_class_counts}; "
        f"expected {expected_testing_class_counts}.")

# ---------------------------------------------------------
# 4. Check for missing files and overlap
# ---------------------------------------------------------

# Create a list of any Testing image paths that do not exist
missing_testing_images = [image_pathfor image_path in testing_manifest["image_path"]
    if not Path(image_path).exists()]


# Verify and load the fixed five-fold Training assignments
if not FOLDS_FILE.exists():
    raise FileNotFoundError(f"Training fold-assignment CSV not found: "f"{FOLDS_FILE}")


assignments = pd.read_csv(FOLDS_FILE)


# Create a set of all relative image paths used for training
training_paths = set(assignments["relative_path"])

# Create a set of all relative image paths used for Testing
testing_paths = set(testing_manifest["relative_path"])

# Find any image paths that appear in both the training and Testing datasets
training_testing_overlap = training_paths & testing_paths

print("\n--- Safety checks ---")
print("Missing Testing images:", len(missing_testing_images))
print("Training/Testing overlap:", len(training_testing_overlap))


# ---------------------------------------------------------
# 5. Stop if validation fails
# ---------------------------------------------------------

if len(testing_manifest) != 1598:
    raise ValueError("Expected 1,598 Testing images, "f"but found {len(testing_manifest)}.")

if testing_manifest["relative_path"].nunique() != 1598:
    raise ValueError("Duplicate Testing image paths were found.")

if missing_testing_images:
    raise FileNotFoundError(f"{len(missing_testing_images)} Testing images are missing.")

if training_testing_overlap:
    raise ValueError("Training and Testing image paths overlap.")

if set(testing_manifest["class"]) != set(CLASS_NAMES):
    raise ValueError("The Testing partition does not contain the expected four classes.")

print("\nHeld-out Testing partition verification passed.")

In [ ]:
# ============================================================
# FINAL EFFICIENTNETB0 EVALUATION ON HELD-OUT TESTING PARTITION
#
# Important:
#   - Loads the previously saved final EfficientNetB0 model
#   - Uses the 1,598 cropped Testing images
#   - Does not train or modify the model
#   - Does not shuffle the Testing data
#   - Uses the same grayscale-to-pseudo-RGB conversion used
#     during EfficientNetB0 training
#   - Retains float32 values in the 0-255 range because
#     EfficientNetB0 contains its own input rescaling
# ============================================================

import gc
import json
import shutil
import time
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
    cohen_kappa_score, confusion_matrix, f1_score, log_loss, matthews_corrcoef,
    precision_score, recall_score)


# ============================================================
# 1. Configuration
# ============================================================

RANDOM_SEED = 42

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

CLASS_NAMES = ("glioma", "meningioma", "notumor", "pituitary")

# Asigning an index to each one of the four classes
CLASS_TO_INDEX = {class_name: index for index, class_name in enumerate(CLASS_NAMES)}

# For each class asign an index
INDEX_TO_CLASS = {index: class_name for class_name, index in CLASS_TO_INDEX.items()}

# Creating the paths for all the folders
PROJECT_ROOT = Path("/content/brain-tumour-mri-classification")

DRIVE_ROOT = Path("/content/drive/MyDrive/brain_tumour_colab")

TESTING_DIRECTORY = (PROJECT_ROOT / "processed_data_cropped" / "Testing")

FINAL_TRAINING_DIRECTORY = ( DRIVE_ROOT / "results"
    / "efficientnetb0_transfer_learning"
    / "efficientnetb0_selected_final_no_augmentation_v2")

FINAL_MODEL_PATH = (FINAL_TRAINING_DIRECTORY / "final_efficientnetb0_cropped.keras")

# Create the path to the marker confirming that final model training finished successfully
FINAL_TRAINING_COMPLETION_PATH = (FINAL_TRAINING_DIRECTORY / "training_complete.json")

# Create the path to the saved configuration used to train the final model
FINAL_TRAINING_CONFIGURATION_PATH = (FINAL_TRAINING_DIRECTORY / "final_training_configuration.json")

EVALUATION_DIRECTORY = (FINAL_TRAINING_DIRECTORY / "testing_evaluation")

PREDICTIONS_PATH = (EVALUATION_DIRECTORY / "testing_predictions.csv")

CONFUSION_MATRIX_PATH = (EVALUATION_DIRECTORY / "confusion_matrix.csv")

CLASSIFICATION_REPORT_CSV_PATH = (EVALUATION_DIRECTORY / "classification_report.csv")

CLASSIFICATION_REPORT_JSON_PATH = (EVALUATION_DIRECTORY / "classification_report.json")

METRICS_PATH = (EVALUATION_DIRECTORY / "testing_metrics.json")

TIMING_PATH = (EVALUATION_DIRECTORY / "testing_timing.json")

TESTING_MANIFEST_PATH = (EVALUATION_DIRECTORY / "testing_manifest.csv")

PROBABILITIES_PATH = (EVALUATION_DIRECTORY / "testing_probabilities.npz")

COMPLETION_PATH = (EVALUATION_DIRECTORY / "evaluation_complete.json")


EXPECTED_CLASS_COUNTS = {
    "glioma": 400,
    "meningioma": 400,
    "notumor": 398,
    "pituitary": 400,
}



# ============================================================
# 2. Utility functions
# ============================================================

# Creating a reusable function for saving data as a JSON file
def save_json_atomic(data, destination):
    """Save JSON through a temporary file."""

    # Converting the destination file location into a Path object to be used later
    destination = Path(destination)

    # Creating a temporary file path used for safety reasons in case something goes wrong with the saving
    temporary_path = destination.with_name(destination.name + ".tmp")

    # Opening the temporary file and writing the data into a JSON file
    with temporary_path.open("w", encoding="utf-8",) as file:
      json.dump(data, file, indent=4, default=str)

    # Replace the destination file with the completed temporary file
    temporary_path.replace(destination)

# Similar function to the previous one saving a dataframe as a CSV file
def save_dataframe_atomic(dataframe, destination):
    """Save a DataFrame through a temporary CSV file."""

    destination = Path(destination)

    temporary_path = destination.with_name(destination.name + ".tmp")

    dataframe.to_csv(temporary_path, index=True)

    temporary_path.replace(destination)


def load_and_preprocess_testing_image(image_path):
    """
    Load one grayscale PNG, repeat its channel three times and convert it to float32.

    EfficientNetB0 retains values in the 0-255 range because its Keras implementation contains an internal rescaling layer.
    """
    # Read the image file from its path
    image_bytes = tf.io.read_file(image_path)

    # Decode the PNG image as a single-channel grayscale image
    image = tf.io.decode_png(image_bytes, channels=1)

    # Confirm the grayscale image has the expected height, width, and 1 channel
    image = tf.ensure_shape(image, (IMAGE_HEIGHT, IMAGE_WIDTH, 1))

    # Convert the grayscale image into a 3-channel RGB image
    image = tf.image.grayscale_to_rgb(image)

    # Convert the image pixel values to float32
    image = tf.cast(image, tf.float32)

    # Confirm the final image has the expected height, width, and 3 channels
    image = tf.ensure_shape(image, (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))

    return image



# ============================================================
# 3. Reproducibility
# ============================================================

np.random.RANDOM_SEED(RANDOM_SEED)
tf.keras.utils.set_random_RANDOM_SEED(RANDOM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
    deterministic_status = "enabled"

except Exception as error:
    deterministic_status = ("requested, but TensorFlow returned: "f"{error}")


# ============================================================
# 4. Path and completion checks
# ============================================================

print("=" * 70)
print("FINAL EFFICIENTNETB0 TESTING EVALUATION")
print("=" * 70)
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("Testing directory:", TESTING_DIRECTORY)
print("Final model:", FINAL_MODEL_PATH)
print("Evaluation directory:", EVALUATION_DIRECTORY)
print("Data augmentation:", False)
print("Testing data shuffled:", False)
print("Deterministic TensorFlow operations:", deterministic_status)


# Testing for errors
if not TESTING_DIRECTORY.exists():
    raise FileNotFoundError("The cropped Testing directory was not found:\n"f"{TESTING_DIRECTORY}")


if not FINAL_MODEL_PATH.exists():
    raise FileNotFoundError("The final trained EfficientNetB0 model was not found:\n"f"{FINAL_MODEL_PATH}")


# Stop if final training did not reach its successful completion marker
if not FINAL_TRAINING_COMPLETION_PATH.exists():
    raise RuntimeError("The final EfficientNetB0 training completion marker was not found:\n"
        f"{FINAL_TRAINING_COMPLETION_PATH}\n"
        "Do not evaluate the Testing partition until final training has completed successfully.")


# Stop if the saved final-training configuration cannot be found
if not FINAL_TRAINING_CONFIGURATION_PATH.exists():
    raise FileNotFoundError("The final EfficientNetB0 training configuration was not found:\n"
        f"{FINAL_TRAINING_CONFIGURATION_PATH}")


# Load the configuration used to train the selected final EfficientNetB0 model
with open(FINAL_TRAINING_CONFIGURATION_PATH, "r", encoding="utf-8") as file:
    final_training_configuration = json.load(file)


# Use the batch size selected during cross-validation and used for final training
BATCH_SIZE = int(final_training_configuration["batch_size"])


print("Testing batch size:", BATCH_SIZE)


if COMPLETION_PATH.exists():
    raise RuntimeError("A completed Testing evaluation already exists:\n"f"{COMPLETION_PATH}\n\n"
        "The cell stopped to prevent accidental re-evaluation or overwriting.")


# Remove only an incomplete Testing evaluation directory.
if EVALUATION_DIRECTORY.exists():
    print("\nRemoving incomplete Testing evaluation output from an earlier attempt...")
    shutil.rmtree( EVALUATION_DIRECTORY)


EVALUATION_DIRECTORY.mkdir(parents=True, exist_ok=True)



# ============================================================
# 5. Enumerate Testing images
# ============================================================


testing_paths = []
testing_labels = []
testing_class_names = []
testing_relative_paths = []
class_counts = {}

# Iterating over the classes
for class_name in CLASS_NAMES:
    class_directory = (TESTING_DIRECTORY / class_name)

    if not class_directory.exists():
        raise FileNotFoundError("Testing class directory not found: "f"{class_directory}")

    # Find and sort all PNG image paths in the current Testing class folder
    class_image_paths = sorted(class_directory.glob("*.png"))

    # Store the number of Testing images found for the current class
    class_counts[class_name] = len(class_image_paths)

    # Get the expected number of Testing images for the current class
    expected_count = (EXPECTED_CLASS_COUNTS[class_name])

    # Checking for errors in the number of testing images found
    if len(class_image_paths) != expected_count:
        raise RuntimeError(
            f"Expected {expected_count} Testing images "
            f"for {class_name}, but found "
            f"{len(class_image_paths)}.")

    # Using the function defined earlier to assign an index to each class
    class_index = (CLASS_TO_INDEX[class_name])

    # Going over the list of images using which are stored as path objects
    for image_path in class_image_paths:

        testing_paths.append(str(image_path)) # Making the image path a string so it can be used later

        testing_labels.append(class_index)    # Store the numeric true class label for the current Testing image

        testing_class_names.append(class_name) # Storing the class name for every image

        # Store the image path relative to the cropped dataset root
        testing_relative_paths.append(image_path.relative_to(PROJECT_ROOT / "processed_data_cropped").as_posix())


# Convert the Testing image paths list into a NumPy string array
testing_paths = np.asarray(testing_paths, dtype=str)

# Convert the Testing labels list into a NumPy int32 array
testing_labels = np.asarray(testing_labels, dtype=np.int32)

# Convert the Testing class names list into a NumPy string array
testing_class_names = np.asarray(testing_class_names, dtype=str)

# Convert the Testing relative paths list into a NumPy string array
testing_relative_paths = np.asarray(testing_relative_paths, dtype=str)

# Testing for errors
if len(testing_paths) != 1598:
    raise RuntimeError("Expected 1,598 cropped Testing images, "f"but found {len(testing_paths)}.")


if len(testing_paths) != len(testing_labels):
    raise RuntimeError("The number of Testing labels does not match the number of Testing paths.")


if len(np.unique(testing_paths)) != 1598:
    raise RuntimeError("Duplicate Testing image paths were found.")


print("\n--- Testing data ---")
print("Testing images:", len(testing_paths))
print("Class mapping:", CLASS_TO_INDEX)

for class_name in CLASS_NAMES:
    print(f"{class_name:<12}: "f"{class_counts[class_name]}")

# Creating the testing manifest dataframe
testing_manifest = pd.DataFrame(
    {
        "relative_path": testing_relative_paths,
        "true_class": testing_class_names,
        "true_index": testing_labels,
    })

# Saving the dataframe as a CSV file
testing_manifest.to_csv(TESTING_MANIFEST_PATH, index=False)


# ============================================================
# 6. Create deterministic Testing dataset
# ============================================================

# Create a TensorFlow dataset from the Testing image paths
testing_dataset = tf.data.Dataset.from_tensor_slices(testing_paths)


# Create an options object which contains instruction for configuring the Testing dataset
dataset_options = (tf.data.Options())


# Keeping the dataset processing order deterministic and reproducible
dataset_options.experimental_deterministic = (True)


# Apply the configured dataset options to the Testing dataset
testing_dataset = (testing_dataset.with_options(dataset_options))


# Load and preprocess every Testing image in the TensorFlow dataset
testing_dataset = testing_dataset.map(
    load_and_preprocess_testing_image,
    num_parallel_calls=tf.data.AUTOTUNE,
    deterministic=True)


# Group the Testing images into batches while keeping the final incomplete batch
testing_dataset = testing_dataset.batch(BATCH_SIZE, drop_remainder=False)


# Cache the processed Testing images and prepare upcoming batches in advance for faster evaluation
cached_testing_dataset = (testing_dataset.cache().prefetch(tf.data.AUTOTUNE))


# ============================================================
# 7. Load the final trained model
# ============================================================

# Go through large variables that may still remain in memory from earlier cells
for variable_name in (
    "model",
    "base_model",
    "final_model",
    "efficientnetb0_model",
    "efficientnetb0_base",
    "final_base_model",
    "training_dataset",
    "final_training_dataset",
    "sample_images",
    "sample_labels",
):

    # Remove the current variable from the global notebook environment if it exists
    globals().pop(variable_name, None)


# Ask Python to free memory from objects that are no longer being used
gc.collect()

# Clear the current Keras session and remove old model state
tf.keras.backend.clear_session()


# Load the saved final trained model without compiling it
final_model = tf.keras.models.load_model(FINAL_MODEL_PATH, compile=False)


print("\n--- Loaded model verification ---")
print("Model name:", final_model.name)
print("Input shape:", final_model.input_shape)
print("Output shape:", final_model.output_shape)
print("Total parameters:", f"{final_model.count_params():,}")


if tuple(final_model.input_shape[1:]) != (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS):
    raise RuntimeError("Unexpected final-model input shape: "f"{final_model.input_shape}")


if int(final_model.output_shape[-1]) != NUMBER_OF_CLASSES:
    raise RuntimeError("Expected four output probabilities, but the model output shape is "f"{final_model.output_shape}.")


# ============================================================
# 8. Populate preprocessing cache
# ============================================================

print("\n" + "=" * 70)
print("TESTING PREPROCESSING")
print("=" * 70)
print("Decoding and preprocessing all 1,598 Testing images...")

# Record the time when Testing image preprocessing begins
preprocessing_start = (time.perf_counter())

cached_image_count = 0
sample_testing_batch = None

# Go through the dataset one batch at a time
for image_batch in cached_testing_dataset:

    # Save the first Testing image batch for later use
    if sample_testing_batch is None:
        sample_testing_batch = image_batch

    # Add the number of images in the current batch to the total
    cached_image_count += int(image_batch.shape[0])

    # Check that every value in the current batch is finite
    if not bool(tf.reduce_all(tf.math.is_finite(image_batch)).numpy()):
        raise RuntimeError("Non-finite values were detected in the preprocessed Testing images.")

    # Check that every image value stays within the expected 0-255 range
    if float(tf.reduce_min(image_batch).numpy()) < 0.0 or float(tf.reduce_max(image_batch).numpy()) > 255.0:
        raise RuntimeError("EfficientNetB0 Testing inputs fall outside the expected 0-255 value range.")


# Calculate how many seconds Testing image preprocessing took
preprocessing_seconds = time.perf_counter() - preprocessing_start

# Checking for errors
if cached_image_count != 1598:
    raise RuntimeError("Expected to preprocess 1,598 Testing images, "f"but processed {cached_image_count}.")

if sample_testing_batch is None:
    raise RuntimeError("No Testing image batch was produced.")


# Find the minimum pixel value in the sample Testing batch
sample_minimum = float(tf.reduce_min(sample_testing_batch).numpy())

# Find the maximum pixel value in the sample Testing batch
sample_maximum = float(tf.reduce_max(sample_testing_batch).numpy())


print("Preprocessed images:", cached_image_count)
print("Sample batch shape:", sample_testing_batch.shape)
print("Sample value range:", sample_minimum, "to", sample_maximum)
print("Testing preprocessing time:", round(preprocessing_seconds, 2), "seconds")



# ============================================================
# 9. Warm up the model
# ============================================================

# This warm-up avoids including TensorFlow graph tracing and
# initial GPU setup in the measured inference time.

# Warm up the model using the same prediction method used during timed inference
_ = final_model.predict(sample_testing_batch, verbose=0)



# ============================================================
# 10. Running the final model on all 1,598 Testing images and
#     storing the model's predicted probabilities
# ============================================================

print("\n" + "=" * 70)
print("HELD-OUT TESTING INFERENCE")
print("=" * 70)


inference_start = (time.perf_counter())

# Run the final model on all cached Testing images and store the predicted probabilities
testing_probabilities = final_model.predict(cached_testing_dataset, verbose=1)


# Calculate how many seconds the Testing inference took
inference_seconds = time.perf_counter() - inference_start


# Convert the predicted probabilities into a NumPy float32 array
testing_probabilities = np.asarray(testing_probabilities, dtype=np.float32)


# Running tests
if testing_probabilities.shape != (1598, NUMBER_OF_CLASSES):
    raise RuntimeError("Unexpected prediction matrix shape: "f"{testing_probabilities.shape}")

if not np.isfinite(testing_probabilities).all():
    raise RuntimeError("Non-finite values were detected in the Testing probabilities.")


# Calculate the total predicted probability for each Testing image
probability_sums = testing_probabilities.sum(axis=1)


# Checking if probabilites some to 1
if not np.allclose(probability_sums, 1.0, atol=1e-5):
    raise RuntimeError("Some Testing probability rows do not sum to one.")


# Find the class with the highest predicted probability for each Testing image
predicted_indices = np.argmax(testing_probabilities, axis=1).astype(np.int32)


# Convert the predicted class indices into their class names
predicted_class_names = np.asarray([INDEX_TO_CLASS[int(index)] for index in predicted_indices], dtype=str)


# Find the highest predicted probability for each Testing image
prediction_confidences = np.max(testing_probabilities, axis=1)


# Check whether each predicted class matches the true Testing label
correct_predictions = predicted_indices == testing_labels


# Calculate the average inference time per Testing image in milliseconds
inference_milliseconds_per_image = inference_seconds / len(testing_paths) * 1000


# Calculate the total Testing time including preprocessing and inference
end_to_end_testing_seconds = preprocessing_seconds + inference_seconds


print("\nCached model inference time:", round(inference_seconds, 2), "seconds")
print("Cached inference time per image:", round(inference_milliseconds_per_image, 3), "milliseconds")
print("Preprocessing + inference time:", round(end_to_end_testing_seconds, 2), "seconds")



# ============================================================
# 11. Calculate final Testing metrics
# ============================================================

# Calculate the overall Testing accuracy
accuracy = accuracy_score(testing_labels, predicted_indices)


# Calculate the balanced accuracy across all classes
balanced_accuracy = balanced_accuracy_score(testing_labels, predicted_indices)


# Calculate the macro-averaged precision across all classes
macro_precision = precision_score(testing_labels, predicted_indices, average="macro", zero_division=0)

# Calculate the macro-averaged recall across all classes
macro_recall = recall_score(testing_labels, predicted_indices, average="macro", zero_division=0)


# Calculate the macro-averaged F1 score across all classes
macro_f1 = f1_score(testing_labels, predicted_indices, average="macro", zero_division=0)


# Calculate the weighted precision across all classes
weighted_precision = precision_score(testing_labels, predicted_indices, average="weighted", zero_division=0)


# Calculate the weighted recall across all classes
weighted_recall = recall_score(testing_labels, predicted_indices, average="weighted", zero_division=0)


# Calculate the weighted F1 score across all classes
weighted_f1 = f1_score(testing_labels, predicted_indices, average="weighted", zero_division=0)


# Calculate the Matthews correlation coefficient
matthews_correlation = matthews_corrcoef(testing_labels, predicted_indices)


# Calculate Cohen's kappa score
cohen_kappa = cohen_kappa_score(testing_labels, predicted_indices)


# Calculate the Testing log loss using the predicted probabilities
testing_log_loss = log_loss(testing_labels, testing_probabilities, labels=list(range(NUMBER_OF_CLASSES)))


# Create the confusion matrix for the Testing predictions
confusion = confusion_matrix(testing_labels, predicted_indices, labels=list(range(NUMBER_OF_CLASSES)))

# Classification report
report_dictionary = (classification_report(testing_labels, predicted_indices,
        labels=list(range(NUMBER_OF_CLASSES)),
        target_names=list(CLASS_NAMES),
        output_dict=True,
        zero_division=0))


# Count how many Testing images were classified correctly
number_correct = int(correct_predictions.sum())

# Count how many Testing images were classified incorrectly
number_incorrect = int(len(correct_predictions) - number_correct)


metrics = {
    "model_name":"EfficientNetB0 transfer learning",
    "dataset_variant":"cropped",
    "dataset_partition":"Testing",
    "testing_image_count":int(len(testing_paths)),
    "class_names":list(CLASS_NAMES),
    "class_to_index":CLASS_TO_INDEX,
    "class_counts":class_counts,
    "accuracy":float(accuracy),
    "balanced_accuracy":float(balanced_accuracy),
    "macro_precision":float(macro_precision),
    "macro_recall":float(macro_recall),
    "macro_f1":float(macro_f1),
    "weighted_precision":float(weighted_precision),
    "weighted_recall":float(weighted_recall),
    "weighted_f1":float(weighted_f1),
    "matthews_correlation_coefficient":float(matthews_correlation),
    "cohen_kappa":float(cohen_kappa),
    "multiclass_log_loss":float(testing_log_loss),
    "correct_predictions":number_correct,
    "incorrect_predictions":number_incorrect,
    "mean_prediction_confidence":float(prediction_confidences.mean()),
    "data_augmentation": False,
    "testing_data_shuffled": False,
    "model_modified_during_testing": False,
    "input_value_range":"0-255 float32",
    "tensorflow_version": tf.__version__,
    "evaluated_at": datetime.now().isoformat(),
}



# ============================================================
# 12. Save prediction-level results
# ============================================================


# Creating a dataframe with the predictions
predictions_frame = pd.DataFrame(
    {
        "relative_path": testing_relative_paths,
        "true_class": testing_class_names,
        "true_index": testing_labels,
        "predicted_class": predicted_class_names,
        "predicted_index": predicted_indices,
        "correct": correct_predictions,
        "prediction_confidence": prediction_confidences,
        "probability_glioma": testing_probabilities[:, 0],
        "probability_meningioma": testing_probabilities[:, 1],
        "probability_notumor": testing_probabilities[:, 2],
        "probability_pituitary": testing_probabilities[:, 3],
    }
)

# Saving the predictions as a csv file
predictions_frame.to_csv(PREDICTIONS_PATH, index=False)

# Creating and saving a copy of the model's testing predictions
np.savez_compressed(
    PROBABILITIES_PATH,
    probabilities=testing_probabilities,
    true_labels=testing_labels,
    predicted_labels=predicted_indices,
    relative_paths=testing_relative_paths,
    class_names=np.asarray(CLASS_NAMES, dtype=str))



# ============================================================
# 13. Save confusion matrix and classification report
# ============================================================


# Confusion matrix
confusion_frame = pd.DataFrame(
    confusion,
    index=[f"actual_{class_name}" for class_name in CLASS_NAMES],
    columns=[f"predicted_{class_name}" for class_name in CLASS_NAMES])


# Save the confusion matrix as a CSV file
save_dataframe_atomic(confusion_frame, CONFUSION_MATRIX_PATH)


# Convert the classification report dictionary into a DataFrame
report_frame = pd.DataFrame(report_dictionary).transpose()


# Save the classification report as a CSV file
save_dataframe_atomic(report_frame, CLASSIFICATION_REPORT_CSV_PATH)


# Save the classification report as a JSON file
save_json_atomic(report_dictionary, CLASSIFICATION_REPORT_JSON_PATH)


# Save the Testing evaluation metrics as a JSON file
save_json_atomic(metrics, METRICS_PATH)




# ============================================================
# 14. Save timing information
# ============================================================

timing_information = {
    "testing_preprocessing_seconds": preprocessing_seconds,
    "cached_model_inference_seconds": inference_seconds,
    "cached_model_inference_milliseconds_per_image": inference_milliseconds_per_image,
    "preprocessing_plus_inference_seconds": end_to_end_testing_seconds,

    "timing_notes": {
        "testing_preprocessing_seconds": (
            "Includes PNG decoding, grayscale-to-RGB "
            "channel replication, float32 conversion "
            "and population of the in-memory cache."),

        "cached_model_inference_seconds": (
            "Measured after preprocessing was cached "
            "and after one warm-up batch. It includes "
            "cached dataset iteration, host-to-device "
            "transfer and model forward inference."),

        "preprocessing_plus_inference_seconds": (
            "Sum of Testing preprocessing and cached "
            "model inference."),
    },
}

save_json_atomic(timing_information, TIMING_PATH)


# Save this file last as the completion marker.
completion_information = {
    "status": "completed",
    "model_path": str(FINAL_MODEL_PATH),
    "evaluation_directory": str(EVALUATION_DIRECTORY),
    "testing_images": int(len(testing_paths)),
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "completed_at": datetime.now().isoformat(),
}

save_json_atomic(completion_information, COMPLETION_PATH)


# ============================================================
# 15. Final output
# ============================================================

print("\n" + "=" * 70)
print("FINAL EFFICIENTNETB0 TESTING RESULTS")
print("=" * 70)
print(f"Accuracy:            {accuracy:.6f}")
print(f"Balanced accuracy:   {balanced_accuracy:.6f}")
print(f"Macro precision:     {macro_precision:.6f}")
print(f"Macro recall:        {macro_recall:.6f}")
print(f"Macro F1-score:      {macro_f1:.6f}")
print(f"Weighted F1-score:   {weighted_f1:.6f}")
print(f"MCC:                 {matthews_correlation:.6f}")
print(f"Cohen's kappa:       {cohen_kappa:.6f}")
print(f"Multiclass log loss: {testing_log_loss:.6f}")
print("\nCorrect predictions:", number_correct)
print("Incorrect predictions:", number_incorrect)
print("\nConfusion matrix")
print("Rows = actual classes; columns = predicted classes")
print(confusion_frame)
print("\nClassification report")
print(
    classification_report(
        testing_labels,
        predicted_indices,
        labels=list(range(NUMBER_OF_CLASSES)),
        target_names=list(CLASS_NAMES),
        digits=6,
        zero_division=0))

print("\n--- Timing ---")
print("Testing preprocessing:", round(preprocessing_seconds, 2), "seconds")
print("Cached model inference:", round(inference_seconds, 2), "seconds")
print("Inference per image:", round(inference_milliseconds_per_image, 3), "milliseconds")
print("Preprocessing + inference:", round(end_to_end_testing_seconds, 2), "seconds")
print("\nEvaluation results saved to:")
print(EVALUATION_DIRECTORY)